# Indicadores de seguimiento de la evaluación del PNIE <br> Parte V: Intervenciones de actores involucrados

Autor: Equipo de Evaluación del PNIE

Fecha de creación: 12/09/2025

Fecha de actualización: 27/10/2025

## 01. Configuración del notebook

In [27]:
# Importando librerias
import os
import numpy as np
import pandas as pd
from dbfread import DBF
import re
import unicodedata
from typing import Optional

In [28]:
# Directorio de trabajo
file1 = 'E:/OneDrive - Ministerio de Educación/00. Paolo/Actividades'
file1 = 'D:/OneDrive - Ministerio de Educación/00. Paolo/Actividades'
file2 = '01. Evaluación PNIE/02. IS-PNIE'
file = file1 + '/' + file2
os.chdir(file)

In [29]:
# Configurando el formato
pd.options.mode.chained_assignment = None
pd.options.display.float_format = '{:,.1f}'.format

In [30]:
# Definiendo funciones de conteo
def unique(data, varlist):
    aux = data[varlist]
    a = aux.drop_duplicates().shape[0]
    list = " ".join(varlist)
    print(f'Number of unique values of {list} is {a:,.0f}')
    print(f'Number of records is {data.shape[0]:,.0f}')

def by_unique(data, varlist, var):
    unique(data, varlist)
    list = varlist + var
    aux = data[list]
    a = aux.drop_duplicates().value_counts(var).reset_index().\
    sort_values(by=var)
    print("\n", a)

In [31]:
# Corrección de codigos locales
def codlocal(df, oldvar, newvar):
    df[newvar] = df[oldvar].astype(str).str.zfill(6)

In [32]:
def freq(df, columna, dropna=False, decimales=1):
    """
    Genera una tabla de frecuencias absolutas, relativas (%) y totales.

    Parámetros:
    -----------
    df : DataFrame
        El DataFrame de entrada
    columna : str
        Nombre de la columna a analizar
    dropna : bool, opcional
        Si True, ignora los valores NaN. Default = False
    decimales : int, opcional
        Número de decimales para el porcentaje. Default = 2
    """
    
    # Frecuencias absolutas y relativas
    freq = df[columna].value_counts(dropna=dropna)
    rel = df[columna].value_counts(normalize=True, dropna=dropna) * 100
    
    # Construcción de la tabla
    tabla = pd.DataFrame({
        'Freq': freq,
        '(%)': rel.round(decimales)
    })
    
    # Totales
    total_freq = tabla['Freq'].sum()
    tabla.loc['TOTAL'] = [total_freq, 100]

    # Formateo: Freq con separador de miles y (%) con % explícito
    tabla['Freq'] = tabla['Freq'].apply(lambda x: f"{int(x):,}" if pd.notna(x) else x)
    tabla['(%)'] = tabla['(%)'].apply(lambda x: f"{x:.{decimales}f}%" if x != 100 else "100%")
 
    return print(tabla, '\n')

In [33]:
def _normalize_text(s: pd.Series) -> pd.Series:
    # Quita tildes/diacríticos y pasa a minúsculas
    def _norm(x):
        if x is None:
            return ""
        x = str(x)
        x = unicodedata.normalize("NFKD", x)
        x = "".join(c for c in x if not unicodedata.combining(c))
        return x.lower()
    return s.fillna("").astype(str).map(_norm)

In [34]:
def flag_word_groups_in_columns(
    df: pd.DataFrame,
    cols: list,
    groups: list[list[str]],
    new_col: str = "flag",
    whole_word: bool = True,
    forbidden: list[str] = None
) -> pd.DataFrame:

    if not cols:
        raise ValueError("Debes proporcionar al menos una columna en `cols`.")
    for c in cols:
        if c not in df.columns:
            raise KeyError(f"Columna no encontrada: {c}")

    # normaliza texto de columnas
    proc_cols = [_normalize_text(df[c]) for c in cols]
    text = proc_cols[0]
    for s in proc_cols[1:]:
        text = text.str.cat(s, sep=" ")

    # === 1. Palabras requeridas (AND entre grupos, OR dentro del grupo) ===
    group_masks = []
    for group in groups:
        words = ["".join(
            ch for ch in unicodedata.normalize("NFKD", str(w).lower())
            if not unicodedata.combining(ch)
        ) for w in group if str(w).strip()]
        if not words:
            # grupo vacío -> no aporta; marca como True para no afectar el AND
            group_masks.append(pd.Series(True, index=df.index))
            continue
        inner = "|".join(re.escape(w) for w in words)
        pat = rf"(?u)\b(?:{inner})\b" if whole_word else inner
        group_masks.append(text.str.contains(pat, regex=True, na=False))

    mask = np.logical_and.reduce(group_masks) if group_masks else \
        pd.Series(False, index=df.index)
    
    # === 2. Palabras prohibidas ===
    if forbidden:
        forbidden_words = ["".join(
            ch for ch in unicodedata.normalize("NFKD", str(w).lower())
            if not unicodedata.combining(ch)
        ) for w in forbidden if str(w).strip()]
        inner_forbidden = "|".join(re.escape(w) for w in forbidden_words)
        pat_forbidden = rf"(?u)\b(?:{inner_forbidden})\b" if whole_word \
            else inner_forbidden
        mask_forbidden = text.str.contains(pat_forbidden, regex=True, na=False)

        # si aparece alguna palabra prohibida, fuerza a 0
        mask = mask & ~mask_forbidden

    df[new_col] = mask.astype("uint8")
    return df

In [35]:
def _unique_preserve_order(seq):
    seen = set()
    out = []
    for x in seq:
        if x is None:
            continue
        s = str(x).strip()
        if s == "" or s.lower() in {"nan", "none"}:
            continue
        if s not in seen:
            seen.add(s)
            out.append(s)
    return out

In [36]:
def reshape_gi_cui_fuente(df: pd.DataFrame) -> pd.DataFrame:
    keys = ["codlocal", "anio_culm"]

    gi_cols = [c for c in df.columns if c.startswith("GI")]
    cui_cols = [c for c in df.columns if c.startswith("CUI_")]
    fuente_cols = [c for c in df.columns if c.startswith("fuente_")]

    # 1) Agregación GI: max para todo menos GI3_3 y GI3_4 que se suman
    gi_agg = {}
    for c in gi_cols:
        gi_agg[c] = "sum" if c in {"GI3_3", "GI3_4"} else "max"

    g = df.groupby(keys, dropna=False)
    gi_df = g[gi_cols].agg(gi_agg).reset_index() if gi_cols else g.size().reset_index(name="tmp").drop(columns=["tmp"])

    # 2) CUI concatenado "(sufijo) valor"
    def make_cui_series(group: pd.DataFrame) -> Optional[str]:
        parts = []
        for c in cui_cols:
            sufijo = c.split("CUI_", 1)[1] if "CUI_" in c else c
            col_vals = group[c].dropna().tolist()
            expanded = []
            for v in col_vals:
                s = str(v)
                if " / " in s:
                    expanded.extend([t.strip() for t in s.split(" / ")])
                else:
                    expanded.append(s.strip())
            for v in _unique_preserve_order(expanded):
                parts.append(f"({sufijo}) {v}")
        parts = _unique_preserve_order(parts)
        return " / ".join(parts) if parts else None

    # 3) fuente concatenada
    def make_fuente(group: pd.DataFrame) -> Optional[str]:
        vals = []
        for c in fuente_cols:
            col_vals = group[c].dropna().tolist()
            expanded = []
            for v in col_vals:
                s = str(v)
                if " / " in s:
                    expanded.extend([t.strip() for t in s.split(" / ")])
                else:
                    expanded.append(s.strip())
            vals.extend(expanded)
        vals = _unique_preserve_order(vals)
        return " / ".join(vals) if vals else None

    cui_series = g.apply(make_cui_series).rename("CUI")
    fuente_series = g.apply(make_fuente).rename("fuente")

    meta_df = pd.concat([cui_series, fuente_series], axis=1).reset_index()

    # merge final
    out = gi_df.merge(meta_df, on=keys, how="left")

    # orden de columnas
    ordered_cols = keys + ["CUI", "fuente"] + gi_cols
    ordered_cols = [c for c in ordered_cols if c in out.columns]
    out = out[ordered_cols]

    # forzar numérico en GI
    for c in gi_cols:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors="coerce")

    return out

## 02. UE 118

In [37]:
file = 'data/procesadas/banco_inversiones_criterios.csv'
bi = pd.read_csv(file, dtype={'CODIGO_INVERSION': 'string'})

C:\Users\diplan27.MINEDU\AppData\Local\Temp\ipykernel_12820\2074733584.py:2: DtypeWarning: Columns (136) have mixed types. Specify dtype option on import or set low_memory=False.
  bi = pd.read_csv(file, dtype={'CODIGO_INVERSION': 'string'})


### 2.1 PMESUT

In [20]:
file = 'data/Intervenciones/PNIE al 2025_04.09_final.xlsx'
df_0 = pd.read_excel(file, sheet_name='PMESUT')

In [21]:
df = df_0.copy()

# Selecccionando información
df.columns = df.iloc[0]
df = df.iloc[1:-1, 1:]

# Filtrando CUI vinculados a locales educativos (no universitarios)
cuis = [2411125, 2234830]
df = df[df['Código Único de Inversiones (CUI)'].isin(cuis)]

# Editando CUI
df.rename({'Código Único de Inversiones (CUI)': 'CUI'}, axis=1, inplace=True)
df['CUI'] = df['CUI'].astype('string')

# Completando información del código del local educativo
df['codlocal'] = ""
df.loc[df['CUI'] == '2411125', 'codlocal'] = '121323'
df.loc[df['CUI'] == '2234830', 'codlocal'] = '220144'

# Identificando intervenciones culminadas
freq(df, 'Estado de la intervención')
df = df.loc[df['Estado de la intervención'] == 'Culminado', :]

# Obteniendo información del BI
df = pd.merge(df, bi,
              how='left',
              left_on='CUI', right_on='CODIGO_INVERSION')
freq(df, 'ETAPA_F8')
freq(df, 'AVANCE_EJECUCION_F12B')
freq(df, 'AVANCE_FISICO_F12B')
freq(df, 'AVANCE_FISICO_F9')
freq(df, 'CERRADO_F9')
freq(df, 'CULMINADA_F9')

print('''\nCONCLUSIÓN: 
      SE DETERMINA QUE LA INTERVENCIÓN NO A CONCLUIDO INTEGRAMENTE''')

                          Freq    (%)
Estado de la intervención            
Culminado                    1  50.0%
En ejecución                 1  50.0%
TOTAL                        2   100% 

                     Freq   (%)
ETAPA_F8                       
Ejecución física (C)    1  100%
TOTAL                   1  100% 

                      Freq   (%)
AVANCE_EJECUCION_F12B           
0.8                      1  100%
 TOTAL                   1  100% 

                   Freq   (%)
AVANCE_FISICO_F12B           
0.8                   1  100%
 TOTAL                1  100% 

                 Freq   (%)
AVANCE_FISICO_F9           
0.0                 1  100%
 TOTAL              1  100% 

           Freq   (%)
CERRADO_F9           
NO            1  100%
TOTAL         1  100% 

             Freq   (%)
CULMINADA_F9           
NO              1  100%
TOTAL           1  100% 


CONCLUSIÓN: 
      SE DETERMINA QUE LA INTERVENCIÓN NO A CONCLUIDO INTEGRAMENTE


### 2.2 PMESTP

In [22]:
file = 'data/Intervenciones/PNIE al 2025_04.09_final.xlsx'
df_0 = pd.read_excel(file, sheet_name='PMESTP')

In [23]:
df = df_0.copy()

# Selecccionando información
df.columns = df.iloc[0]
df = df.iloc[1:-1, 1:]

# Filtrando CUI vinculados a locales educativos (no universitarios)
cuis = [2553078, 2538005, 2542652, 2506626, 2475486, 2475185, 2432205]
df = df[df['Código Único de Inversiones (CUI)'].isin(cuis)]

# Editando CUI
df.rename({'Código Único de Inversiones (CUI)': 'CUI'}, axis=1, inplace=True)
df['CUI'] = df['CUI'].astype('string')

# Completando información del código del local educativo
df['codlocal'] = ""
df.loc[df['CUI'] == '2506626', 'codlocal'] = '171879'
df.loc[df['CUI'] == '2553078', 'codlocal'] = '394610'
df.loc[df['CUI'] == '2542652', 'codlocal'] = '149666'
df.loc[df['CUI'] == '2538005', 'codlocal'] = '445087'
df.loc[df['CUI'] == '2475486', 'codlocal'] = '394653'
df.loc[df['CUI'] == '2475185', 'codlocal'] = '860882'
df.loc[df['CUI'] == '2432205', 'codlocal'] = '490447'

# Identificando intervenciones culminadas
freq(df, 'Estado de la intervención')
# df = df.loc[df['Estado de la intervención'] == 'Culminado', :]

# Obteniendo información del BI
df = pd.merge(df, bi,
              how='left',
              left_on='CUI', right_on='CODIGO_INVERSION')
freq(df, 'ETAPA_F8')
freq(df, 'AVANCE_EJECUCION_F12B')
freq(df, 'AVANCE_FISICO_F12B')
freq(df, 'AVANCE_FISICO_F9')
freq(df, 'CERRADO_F9')
freq(df, 'CULMINADA_F9')

print('''\nCONCLUSIÓN: 
      SE DETERMINA QUE LA INTERVENCIÓN NO A CONCLUIDO INTEGRAMENTE''')

                                                   Freq    (%)
Estado de la intervención                                     
Proceso de selección DESIERTO                         4  57.1%
Proyecto adjudicado, en proceso de firma              2  28.6%
En evaluación del comité de selección. Opinión ...    1  14.3%
TOTAL                                                 7   100% 

                 Freq   (%)
ETAPA_F8                   
Consistencia (A)    7  100%
TOTAL               7  100% 

                      Freq   (%)
AVANCE_EJECUCION_F12B           
0.0                      7  100%
 TOTAL                   7  100% 

                   Freq   (%)
AVANCE_FISICO_F12B           
0.0                   7  100%
 TOTAL                7  100% 

                 Freq   (%)
AVANCE_FISICO_F9           
0.0                 7  100%
 TOTAL              7  100% 

           Freq   (%)
CERRADO_F9           
NO            7  100%
TOTAL         7  100% 

             Freq   (%)
CULMINADA_F9          

## 03. PEIP-EB

In [38]:
file = 'data/Intervenciones/PEIP_2507.dta'
df_0 = pd.read_stata(file)

In [39]:
df = df_0.copy()

# Editando CUI y codlocal
codlocal(df, 'cod_local', 'codlocal')
df.rename({'cui': 'CUI'}, axis=1, inplace=True)
df['CUI'] = df['CUI'].astype('string')

# Obteniendo información del BI
df = pd.merge(df, bi,
               how='left',
               left_on='CUI', right_on='CODIGO_INVERSION')

freq(df, 'cf')

# Seleccionando variables
df = df[['codlocal', 'CUI_x', 'y_obra']]
df.rename(columns={'CUI_x': 'CUI',
                   'y_obra': 'anio_culm'}, inplace=True)

# Exportando data
df.to_csv('data/procesadas/peip_intervenciones.csv', index=False)

      Freq   (%)
cf              
1       45  100%
TOTAL   45  100% 



## 04. PRONIED

In [50]:
file = 'data/procesadas/banco_inversiones_criterios.csv'
bi = pd.read_csv(file, dtype={'CODIGO_INVERSION': 'string'})

C:\Users\Paolo\AppData\Local\Temp\ipykernel_23816\2074733584.py:2: DtypeWarning: Columns (136) have mixed types. Specify dtype option on import or set low_memory=False.
  bi = pd.read_csv(file, dtype={'CODIGO_INVERSION': 'string'})


### 4.1 UGEO

In [51]:
file1 = 'data/Intervenciones/'
file2 = 'Anexo 01 - Relación de Intervenciones UGEO 2017-2025.xlsx'
df_0 = pd.read_excel(file1 + file2)

In [ ]:
df = df_0.copy()

# Seleccionando información
df.columns = df.iloc[1, :]
df = df[2:].reset_index(drop=True)

df.rename({'Código Único de Inversión (CUI)': 'CUI'}, axis=1, inplace=True)
df['CUI'] = df['CUI'].astype('string')
codlocal(df, 'Código del local educativo', 'codlocal')

# Corrigiendo IOARR
df['activo'] = df['Nombre de la institución educativa'].str.split('-'). \
    str[-1].str.strip()
df['ind'] = ''
df.loc[
    (df['Tipo de inversión\n(proyecto de inversión regular, IOARR u OXI)'] == \
        'IOARR') & (df['activo'] == 'RIN'), 'ind'] = 'GI1_2'
cp = ['CERCO', 'CERCO PERIMETRICO']
df.loc[
    (df['Tipo de inversión\n(proyecto de inversión regular, IOARR u OXI)'] == \
        'IOARR') & (df['activo'].isin(cp)), 'ind'] = 'GI1_5'

# Resto de intervenciones
df.loc[df['ind'] == '', 'ind'] = 'sust_total'

# Obteniendo información sobre LLEE nuevos y existente
exis = pd.read_csv('data/procesadas/llee_existentes.csv', 
                   dtype={'codlocal': str})
df = pd.merge(df, exis,
              on='codlocal', how='left', indicator='existente')

# Elaborando los indicadores del GI5
df.loc[(df['ind'] == 'sust_total') & (df['existente'] == 'both'),
       'ind'] = 'GI5_1'
df.loc[(df['ind'] == 'sust_total') & (df['existente'] == 'left_only'),
       'ind'] = 'GI5_2'

# Año de culminación
df['fec_culm'] = pd.to_datetime(df['Fecha de culminación de obra (o estimada)'], 
                                errors='coerce')
df['anio_culm'] = df['fec_culm'].dt.year.astype('Int64', errors='ignore')

# Eliminando duplicidad
df = df.sort_values(['codlocal', 'CUI', 'ind', 'anio_culm'], 
                    ascending=[True, True, True, False])
df = df.drop_duplicates(subset=['codlocal', 'CUI', 'ind'], keep='first'). \
    reset_index(drop=True)
df.to_csv('data/procesadas/pronied_ugeo_pre.csv', index=False)
df = (
    df.groupby(['codlocal', 'ind'], as_index=False)
      .agg({
          'CUI': lambda x: " / ".join(sorted(x.astype(str).unique())),
          'anio_culm': 'max'
      })
)

# Corrección de codlocal
# df.loc[df['codlocal'] == '344093', 'codlocal'] = '344130'

# Exportando data
freq(df, 'ind')
freq(df, 'anio_culm')
unique(df, ['codlocal'])
df = df[['codlocal', 'CUI', 'ind', 'anio_culm']]
df.to_csv('data/procesadas/pronied_ugeo.csv', index=False)

      Freq    (%)
ind              
GI5_1  137  82.5%
GI1_5   16   9.6%
GI1_2    7   4.2%
GI5_2    6   3.6%
TOTAL  166   100% 

          Freq    (%)
anio_culm            
2017        36  21.7%
2018        30  18.1%
2021        24  14.5%
2020        18  10.8%
2019        16   9.6%
2023        14   8.4%
2022        11   6.6%
2016         9   5.4%
2024         7   4.2%
2014         1   0.6%
TOTAL      166   100% 

Number of unique values of codlocal is 166
Number of records is 166


### 4.2 UGRD

#### 4.2.1 MBR

In [53]:
file1 = 'data/Intervenciones/'
file2 = 'Anexo 02 - Relación de inversiones del PIRCC.xlsx'
df_0 = pd.read_excel(file1 + file2)

In [54]:
df = df_0.copy()

# Seleccionando información
df.columns = df.iloc[1, :]
df = df[2:].reset_index(drop=True)

df['FUR'] = df['FUR'].astype('string')
codlocal(df, 'Código del local educativo', 'codlocal')

# Seleccionando MBR
freq(df, 'Tipo de intervención\n(MBR o ME)')
df = df.loc[df['Tipo de intervención\n(MBR o ME)'] == 'MBR', :]

# Criterios para la selección de intervenciones
# i. Año de culminación hasta 2025
df['fec_culm'] = pd.to_datetime(df['Fecha de culminación de obra (o estimada)'], 
                                errors='coerce')
df['anio_culm'] = df['fec_culm'].dt.year.astype('Int64', errors='ignore')
df.loc[df['anio_culm'] == 1970, 'anio_culm'] = \
    df['Fecha de culminación de obra (o estimada)']
df['cf1'] = np.where(df['anio_culm'] <= 2025, 1, 0)
freq(df, 'cf1')

# ii. Avance fisico mayor a 85%
df['cf2'] = np.where(df['Avance físico\n(%)'] >= 0.85, 1, 0)
freq(df, 'cf2')

# iii. Selccionando intervenciones que cumplen con los criterios
df['cf'] = np.where((df['cf1'] == 1) & (df['cf2'] == 1), 1, 0)
freq(df, 'cf')
df = df.loc[df['cf'] == 1, :]

# Obteniendo información sobre LLEE nuevos y existente
exis = pd.read_csv('data/procesadas/llee_existentes.csv', 
                   dtype={'codlocal': str})
df = pd.merge(df, exis,
              on='codlocal', how='left', indicator='existente')

# Elaborando los indicadores del GI5
df['ind'] = ''
df.loc[(df['existente'] == 'both'), 'ind'] = 'GI5_1'
df.loc[(df['existente'] == 'left_only'), 'ind'] = 'GI5_2'

# Exportando data
freq(df, 'ind')
unique(df, ['codlocal'])
df = df[['codlocal', 'FUR', 'ind', 'anio_culm']]
df.to_csv('data/procesadas/pronied_ugrd_mbr.csv', index=False)
freq(df, 'anio_culm')

                                 Freq    (%)
Tipo de intervención\n(MBR o ME)            
ME                                312  78.2%
MBR                                87  21.8%
TOTAL                             399   100% 

      Freq    (%)
cf1              
1       59  67.8%
0       28  32.2%
TOTAL   87   100% 

      Freq    (%)
cf2              
1       59  67.8%
0       28  32.2%
TOTAL   87   100% 

      Freq    (%)
cf               
1       55  63.2%
0       32  36.8%
TOTAL   87   100% 

      Freq    (%)
ind              
GI5_1   47  85.5%
GI5_2    8  14.5%
TOTAL   55   100% 

Number of unique values of codlocal is 55
Number of records is 55
          Freq    (%)
anio_culm            
2023        18  32.7%
2024        14  25.5%
2022        11  20.0%
2025        10  18.2%
2021         2   3.6%
TOTAL       55   100% 



#### 4.2.2 ME

In [55]:
file1 = 'data/Intervenciones/'
file2 = 'Anexo 02 - Relación de inversiones del PIRCC.xlsx'
df_0 = pd.read_excel(file1 + file2)

In [56]:
df = df_0.copy()

# Seleccionando información
df.columns = df.iloc[1, :]
df = df[2:].reset_index(drop=True)

df['FUR'] = df['FUR'].astype('string')
codlocal(df, 'Código del local educativo', 'codlocal')

# Seleccionando MBR
freq(df, 'Tipo de intervención\n(MBR o ME)')
df = df.loc[df['Tipo de intervención\n(MBR o ME)'] == 'ME', :]

# Criterios para la selección de intervenciones
# i. Año de culminación hasta 2025
df['fec_culm'] = pd.to_datetime(df['Fecha de culminación de obra (o estimada)'], 
                                errors='coerce')
df['anio_culm'] = df['fec_culm'].dt.year.astype('Int64', errors='ignore')
df.loc[df['anio_culm'] == 1970, 'anio_culm'] = \
    df['Fecha de culminación de obra (o estimada)']
df['cf1'] = np.where(df['anio_culm'] <= 2025, 1, 0)
freq(df, 'cf1')

# ii. Avance fisico mayor a 85%
df['cf2'] = np.where(df['Avance físico\n(%)'] >= 0.85, 1, 0)
freq(df, 'cf2')

# iii. Seleccionando intervenciones que cumplen con los criterios
df['cf'] = np.where((df['cf1'] == 1) & (df['cf2'] == 1), 1, 0)
freq(df, 'cf')
df = df.loc[df['cf'] == 1, :]

# Identificando indicadores
# freq(df, df.columns[14])
# freq(df, df.columns[15])
df['GI1_1'] = np.where(df.iloc[:, 15] == 'SI', 1, 0)

# freq(df, df.columns[16])
# freq(df, df.columns[17])
# freq(df, df.columns[18])
df['GI1_5'] = np.where(df.iloc[:, 18] == 'SI', 1, 0)

# freq(df, df.columns[19])
df['GI4_1'] = np.where(df.iloc[:, 19] == 'SI', 1, 0)

# freq(df, df.columns[20])
# freq(df, df.columns[21])
df['GI3_2'] = np.where(df.iloc[:, 21] == 'SI', 1, 0)

# freq(df, df.columns[22])
df['GI2_1'] = np.where(df.iloc[:, 22] == 'SI', 1, 0)

# freq(df, df.columns[23])
# freq(df, df.columns[24])
# freq(df, df.columns[25])
df['GI4_2'] = np.where(df.iloc[:, 25] == 'SI', 1, 0)

# freq(df, df.columns[26])

# Exportando data
freq(df, 'anio_culm')
unique(df, ['codlocal'])
df = df[['codlocal', 'FUR', 'anio_culm',
         'GI1_1', 'GI1_5', 'GI4_1', 'GI3_2', 'GI2_1', 'GI4_2']]
df.to_csv('data/procesadas/pronied_ugrd_me.csv', index=False)

                                 Freq    (%)
Tipo de intervención\n(MBR o ME)            
ME                                312  78.2%
MBR                                87  21.8%
TOTAL                             399   100% 

      Freq    (%)
cf1              
0      280  89.7%
1       32  10.3%
TOTAL  312   100% 

      Freq    (%)
cf2              
0      295  94.6%
1       17   5.4%
TOTAL  312   100% 

      Freq    (%)
cf               
0      297  95.2%
1       15   4.8%
TOTAL  312   100% 

          Freq    (%)
anio_culm            
2019        10  66.7%
2025         2  13.3%
2021         1   6.7%
2022         1   6.7%
2023         1   6.7%
TOTAL       15   100% 

Number of unique values of codlocal is 15
Number of records is 15


### 4.3 UGME

#### 4.3.1 Sistemas modulares

In [57]:
file1 = 'data/Intervenciones/'
file2 = 'Anexo 3 - UGME.xlsx'
df_0 = pd.read_excel(file1 + file2, sheet_name='Sistemas modulares')

In [58]:
df = df_0.copy()

# Seleccionando información
df.columns = df.iloc[1, :]
df = df[2:].reset_index(drop=True)

# Corrigiendo código de local
codlocal(df, 'Código del local educativo', 'codlocal')
nocod = ['No tiene']
df = df[~df['codlocal'].isin(nocod)]

# Año de culminación
df['fec_culm'] = pd.to_datetime(
    df['Fecha de entrega de los módulos (o estimada)'], 
    errors='coerce')
df['anio_culm'] = df['fec_culm'].dt.year.astype('Int64', errors='ignore')
df = df.loc[df['anio_culm'] <= 2025, :]

# Indicadores
# No se consideran kits de pararrayos
df = df[df['Descripción del bien (tipologías)'] != 'KIT DE PARARRAYO']

# Acceso a agua y alcantarillado
df['acceso_agua'] = np.where(
    df['Descripción del bien (tipologías)'] == 'KIT DE RED DE AGUA', 1, 0)
df['GI2_2'] = np.where(
    df['Descripción del bien (tipologías)'] == 'KIT DE RED DE AGUA', 1, 0)
df['acceso_alcantarillado'] = np.where(
    df['Descripción del bien (tipologías)'] == 'KIT DE RED DE DESAGUE', 1, 0)
df['GI2_1'] = np.where(
    (df['acceso_agua'] == 1) & (df['acceso_alcantarillado'] == 1), 1, 0
)

# Módulos de SS.HH. (GI2.2)
sshh = ['KIT BAÑOS - A', 'MODULO PREFABRICADO DE SS.HH. TIPO COSTA',
        'MODULO PREFABRICADO DE SS.HH. TIPO HELADAS',
        'MODULO PREFABRICADO DE SS.HH. TIPO SELVA',
        'MODULO PREFABRICADO DE SS.HH. TIPO SIERRA']
df.loc[df['Descripción del bien (tipologías)'].isin(sshh), 'GI2_2'] = 1

# Módulos Plan Selva
df['GI1_4'] = np.where(
    df['¿La intervención implicó la instalación de módulos tipo Plan Selva?\n(SÍ / NO)'] == 'SI',
    1, 0)

# Instalación de aulas provisionales
df['aux'] = df[['acceso_agua', 'acceso_alcantarillado', 'GI2_1', 'GI2_2', 
                'GI1_4']].sum(axis=1)
df['GI1_1'] = np.where(df['aux'] == 0, 1, 0)
df.loc[df['GI1_4'] == 1, 'GI1_1'] = 1

# Colapsando a nivel de local educativo y año de culminación
df = df.groupby(['codlocal', 'anio_culm'], as_index=False).agg({
    'GI1_1': 'max',
    'GI1_4': 'max',
    'GI2_1': 'max',
    'GI2_2': 'max'
})

# Exportando data
freq(df, 'anio_culm')
unique(df, ['codlocal'])
unique(df, ['codlocal', 'anio_culm'])
df.to_csv('data/procesadas/pronied_ugme_sist_modulares.csv', index=False)

            Freq    (%)
anio_culm              
2021         542  18.8%
2017         514  17.8%
2019         422  14.7%
2022         342  11.9%
2018         340  11.8%
2020         285   9.9%
2025         169   5.9%
2023         156   5.4%
2024         110   3.8%
TOTAL      2,880   100% 

Number of unique values of codlocal is 2,674
Number of records is 2,880
Number of unique values of codlocal anio_culm is 2,880
Number of records is 2,880


#### 4.3.2 Mobiliario y equipamiento

Se eliminaron 7 intervenciones:
['000002', '000003', '3301315', '3950405', '0547458', '0592746',
        '0657111']

In [59]:
file1 = 'data/Intervenciones/'
file2 = 'Anexo 3 - UGME.xlsx'
df_0 = pd.read_excel(file1 + file2, sheet_name='Mobiliario y equipamiento')

In [60]:
df = df_0.copy()

# Seleccionando información
df.columns = df.iloc[1, :]
df = df[2:].reset_index(drop=True)

# Corrigiendo codlocal y CUI
df.rename({'Código Único de Inversión (CUI)': 'CUI'}, axis=1, inplace=True)
df['CUI'] = df['CUI'].astype('string')
codlocal(df, 'Código del local educativo', 'codlocal')

# Año de culminación
df['fec_culm'] = pd.to_datetime(
    df['Fecha de entrega de los bienes (o estimada)'], 
    errors='coerce')
df['anio_culm'] = df['fec_culm'].dt.year.astype('Int64', errors='ignore')
df = df.loc[df['anio_culm'] <= 2025, :]

# Solucion de la necesidad de mobiliario y equipamiento
freq(df, '¿La intervención resolvió la necesidad del local educativo? (Es decir, luego de la dotación o reposición no se requiere intervención adicional en mobiliario o equipamiento)\n(SÍ / NO)')

# Colapsando a nivel de local educativo, año de culminación y CUI
df = df.groupby(['codlocal', 'CUI', 'Grupo\n(Mobiliario, Equipamiento, etc.)'], 
                as_index=False).agg({
    'anio_culm': 'max',
    'Total de bienes': 'sum'
})

df = df.pivot_table(index=["codlocal", "anio_culm", "CUI"],
                    columns='Grupo\n(Mobiliario, Equipamiento, etc.)',
                    values="Total de bienes", aggfunc="sum").reset_index()
df.columns.name = None; df = df.rename_axis(None, axis=1)
df['EQUIPAMIENTO'] = df['EQUIPAMIENTO'].fillna(0)
df['MOBILIARIO'] = df['MOBILIARIO'].fillna(0)

df.to_csv('data/procesadas/pronied_ugme_me_prev.csv', index=False)

def concat_cui(series):
    return '/'.join(sorted(series.astype(str).unique()))
agg = {'CUI': concat_cui, 'EQUIPAMIENTO': 'sum', 'MOBILIARIO': 'sum'}
df = df.groupby(['codlocal', 'anio_culm'], as_index=False).agg(agg)

# Eliminando intervenciones no escolarizadas
noes = ['000002', '000003', '3301315', '3950405', '0547458', '0592746',
        '0657111']
df = df[~df['codlocal'].isin(noes)]
# Indicador
df['ind'] = ''
df.loc[df['EQUIPAMIENTO'] > 0, 'ind'] = 'GI3_2'
df.loc[df['MOBILIARIO'] > 0, 'ind'] = 'GI3_2'
freq(df, 'ind')

# Exportando data
df.rename({'CUI': 'cui_mob_equ', 'EQUIPAMIENTO': 'ugme_equ_bienes',
           'MOBILIARIO': 'ugme_mob_bienes'}, axis=1, inplace=True)
freq(df, 'ind')
freq(df, 'anio_culm')
unique(df, ['codlocal', 'anio_culm'])
unique(df, ['codlocal'])
df.to_csv('data/procesadas/pronied_ugme_me.csv', index=False)

                                                      Freq   (%)
¿La intervención resolvió la necesidad del loca...              
SI                                                  21,532  100%
TOTAL                                               21,532  100% 

        Freq   (%)
ind               
GI3_2  1,873  100%
TOTAL  1,873  100% 

        Freq   (%)
ind               
GI3_2  1,873  100%
TOTAL  1,873  100% 

            Freq    (%)
anio_culm              
2018         838  44.7%
2017         263  14.0%
2024         259  13.8%
2021         164   8.8%
2020         152   8.1%
2022          97   5.2%
2025          50   2.7%
2019          32   1.7%
2023          18   1.0%
TOTAL      1,873   100% 

Number of unique values of codlocal anio_culm is 1,873
Number of records is 1,873
Number of unique values of codlocal is 1,538
Number of records is 1,873


C:\Users\Paolo\AppData\Local\Temp\ipykernel_23816\1516579976.py:33: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['EQUIPAMIENTO'] = df['EQUIPAMIENTO'].fillna(0)
C:\Users\Paolo\AppData\Local\Temp\ipykernel_23816\1516579976.py:34: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['MOBILIARIO'] = df['MOBILIARIO'].fillna(0)


### 4.4 UGM

#### 4.4.1 Accesibilidad

In [61]:
file1 = 'data/Intervenciones/'
file2 = 'Anexo 4_UGM.xlsx'
df_0 = pd.read_excel(file1 + file2, sheet_name='Accesibilidad (2017-2025)')

In [62]:
df = df_0.copy()

# Seleccionando información
df.columns = df.iloc[1, :]
df = df[2:-7].reset_index(drop=True)
codlocal(df, 'Código del local educativo', 'codlocal')
df.drop(columns=['N°'], inplace=True)
df.drop_duplicates(inplace=True)

# Indicador de accesibilidad
freq(df, df.columns[9])
freq(df, df.columns[10])
df['GI4_3'] = np.where((df[df.columns[9]] == 'SI') | 
                       (df[df.columns[10]] == 'SI'), 1, 0)
freq(df, 'GI4_3')
df = df[df['GI4_3'] == 1]

# Colapsando a nivel de codlocal
df = df[['codlocal', 'Año', 'GI4_3']]
df.rename(columns={'Año': 'anio_culm'}, inplace=True)
df = df.groupby('codlocal', as_index=False).agg({
    'anio_culm': 'max',
    'GI4_3': 'max'
})
# Exportando data
freq(df, 'anio_culm')
unique(df, ['codlocal'])
df.to_csv('data/procesadas/pronied_ugm_accesibilidad.csv', index=False)

                                                     Freq    (%)
¿La intervención implicó actividades de dotació...              
NO                                                  1,824  66.8%
SI                                                    658  24.1%
(****)                                                247   9.1%
TOTAL                                               2,729   100% 

                                                     Freq    (%)
¿La intervención implicó actividades de instala...              
NO                                                  1,785  65.4%
SI                                                    697  25.5%
(****)                                                247   9.1%
TOTAL                                               2,729   100% 

        Freq    (%)
GI4_3              
0      1,848  67.7%
1        881  32.3%
TOTAL  2,729   100% 

          Freq    (%)
anio_culm            
2021       166  24.7%
2023       159  23.7%
2024       133  19.8%
2020

NOTAS:	
- (*)	Para los años 2018 y 2019, el monto corresponde al transferido.
- (**)	Para el año 2018, la fecha de término se considera el último día del mismo año.
- (***)	Durante el año 2017, la DG del programa de Acondicionamiento no consideró la partida de Señalización.
- (****)	No se cuenta con información del detalle en el gasto por partidas para el año 2018.
- (*****)	Si bien fue beneficiario del listado de locales de Acondicionamiento, se le asignó un monto de 0 para estas actividades (solo se le programó presupuesto para útiles escolares y de escritorio).


#### 4.4.2 Mantenimiento preventivo

In [63]:
file = 'data/Intervenciones/Anexo 4_UGM.xlsx'
xls = pd.ExcelFile(file)
xls.sheet_names

['Mto preventivo 2017-I',
 'Mto preventivo 2018-I',
 'Mto preventivo 2019-I',
 'Mto preventivo 2020-0',
 'Mto preventivo 2020-I',
 'Mto preventivo 2020-3',
 'Mto preventivo 2021-I',
 'Mto preventivo 2022-I',
 'Mto preventivo 2023-0',
 'Mto preventivo 2023-I',
 'Mto preventivo 2023-3',
 'Mto preventivo 2024-I',
 'Mto preventivo 2025-I',
 'Accesibilidad (2017-2025)',
 'Notas']

##### 4.4.2.1 2017

In [64]:
file = 'data/Intervenciones/Anexo 4_UGM.xlsx'
df_0 = pd.read_excel(file, sheet_name='Mto preventivo 2017-I')

In [65]:
df = df_0.copy()

# Seleccionando información
df.columns = df.iloc[1, :]
df = df[2:].reset_index(drop=True)

# Seleccionando variables
codlocal(df, 'Código del local educativo', 'codlocal')
df.rename({'Monto incurrido en el mantenimiento\n(S/)': 'monto_mp'},
          axis=1, inplace=True)
df = df[['codlocal', 'monto_mp']]

# Solo LL.EE. con monto
df = df[df['monto_mp'] > 0]
df['monto_mp'] = df['monto_mp'].astype(float)
df.drop_duplicates(inplace=True)
unique(df, ['codlocal'])
mp_2017 = df.copy()

# df['monto_mp'].isna().sum()

Number of unique values of codlocal is 25,463
Number of records is 25,463


##### 4.4.2.2 2018

In [66]:
file = 'data/Intervenciones/Anexo 4_UGM.xlsx'
df_0 = pd.read_excel(file, sheet_name='Mto preventivo 2018-I')

In [67]:
df = df_0.copy()

# Seleccionando información
df.columns = df.iloc[1, :]
df = df[2:].reset_index(drop=True)

# Seleccionando variables
codlocal(df, 'Código del local educativo', 'codlocal')
df.rename({'Monto incurrido en el mantenimiento\n(S/)': 'monto_mp'},
          axis=1, inplace=True)
df = df[['codlocal', 'monto_mp']]

# Solo LL.EE. con monto
df = df[df['monto_mp'] > 0]
df['monto_mp'] = df['monto_mp'].astype(float)
df.drop_duplicates(inplace=True)
unique(df, ['codlocal'])
mp_2018 = df.copy()

# df['monto_mp'].isna().sum()

Number of unique values of codlocal is 34,540
Number of records is 34,540


##### 4.4.2.3 2019

In [68]:
file = 'data/Intervenciones/Anexo 4_UGM.xlsx'
df_0 = pd.read_excel(file, sheet_name='Mto preventivo 2019-I')

In [69]:
df = df_0.copy()

# Seleccionando información
df.columns = df.iloc[1, :]
df = df[2:-3].reset_index(drop=True)

# Seleccionando variables
codlocal(df, 'Código del local educativo', 'codlocal')
df.rename({'Monto incurrido en el mantenimiento\n(S/)\n(*)': 'monto_mp'},
          axis=1, inplace=True)
df = df[['codlocal', 'monto_mp']]

# Solo LL.EE. con monto
df = df[df['monto_mp'] > 0]
df['monto_mp'] = df['monto_mp'].astype(float)
df.drop_duplicates(inplace=True)
unique(df, ['codlocal'])
mp_2019 = df.copy()

# df['monto_mp'].isna().sum()
# df.sort_values('codlocal')

Number of unique values of codlocal is 48,881
Number of records is 48,881


##### 4.4.2.4 2020

###### 2020-0

In [70]:
file = 'data/Intervenciones/Anexo 4_UGM.xlsx'
df_0 = pd.read_excel(file, sheet_name='Mto preventivo 2020-0')

In [71]:
df = df_0.copy()

# Seleccionando información
df.columns = df.iloc[1, :]
df = df[2:].reset_index(drop=True)

# Seleccionando variables
codlocal(df, 'Código del local educativo', 'codlocal')
df.rename({'Monto incurrido en el mantenimiento\n(S/)': 'monto_mp'},
          axis=1, inplace=True)
df = df[['codlocal', 'monto_mp']]

# Solo LL.EE. con monto
df = df[df['monto_mp'] > 0]
df['monto_mp'] = df['monto_mp'].astype(float)
df.drop_duplicates(inplace=True)
unique(df, ['codlocal'])
mp_2020_0 = df.copy()

# print(df['monto_mp'].isna().sum())

Number of unique values of codlocal is 39,993
Number of records is 39,993


###### 2020-1

In [72]:
file = 'data/Intervenciones/Anexo 4_UGM.xlsx'
df_0 = pd.read_excel(file, sheet_name='Mto preventivo 2020-I')

In [73]:
df = df_0.copy()

# Seleccionando información
df.columns = df.iloc[1, :]
df = df[2:].reset_index(drop=True)

# Seleccionando variables
codlocal(df, 'Código del local educativo', 'codlocal')
df.rename({'Monto incurrido en el mantenimiento\n(S/)': 'monto_mp'},
          axis=1, inplace=True)
df = df[['codlocal', 'monto_mp']]

# Solo LL.EE. con monto
df = df[df['monto_mp'] > 0]
df['monto_mp'] = df['monto_mp'].astype(float)
df.drop_duplicates(inplace=True)
unique(df, ['codlocal'])
mp_2020_1 = df.copy()

# print(df['monto_mp'].isna().sum())

Number of unique values of codlocal is 37,871
Number of records is 37,871


###### 2020-3

In [74]:
file = 'data/Intervenciones/Anexo 4_UGM.xlsx'
df_0 = pd.read_excel(file, sheet_name='Mto preventivo 2020-3')

In [75]:
df = df_0.copy()

# Seleccionando información
df.columns = df.iloc[1, :]
df = df[2:].reset_index(drop=True)

# Seleccionando variables
codlocal(df, 'Código del local educativo', 'codlocal')
df.rename({'Monto incurrido en el mantenimiento\n(S/)': 'monto_mp'},
          axis=1, inplace=True)
df = df[['codlocal', 'monto_mp']]

# Solo LL.EE. con monto
df = df[df['monto_mp'] > 0]
df['monto_mp'] = df['monto_mp'].astype(float)
df.drop_duplicates(inplace=True)
unique(df, ['codlocal'])
mp_2020_3 = df.copy()

# print(df['monto_mp'].isna().sum())

Number of unique values of codlocal is 37,971
Number of records is 37,971


###### Consolidacion

In [76]:
mp_2020 = pd.concat([mp_2020_0, mp_2020_1, mp_2020_3],
                    ignore_index=True)
mp_2020 = mp_2020.groupby('codlocal', as_index=False).sum()
unique(mp_2020, ['codlocal'])

Number of unique values of codlocal is 49,830
Number of records is 49,830


##### 4.4.2.5 2021

In [77]:
file = 'data/Intervenciones/Anexo 4_UGM.xlsx'
df_0 = pd.read_excel(file, sheet_name='Mto preventivo 2021-I')

In [78]:
df = df_0.copy()

# Seleccionando información
df.columns = df.iloc[1, :]
df = df[2:].reset_index(drop=True)

# Seleccionando variables
codlocal(df, 'Código del local educativo', 'codlocal')
df.rename({'Monto incurrido en el mantenimiento\n(S/)': 'monto_mp'},
          axis=1, inplace=True)
df = df[['codlocal', 'monto_mp']]

# Solo LL.EE. con monto
df = df[df['monto_mp'] > 0]
df['monto_mp'] = df['monto_mp'].astype(float)
df.drop_duplicates(inplace=True)
unique(df, ['codlocal'])
mp_2021 = df.copy()

# print(df['monto_mp'].isna().sum())
# df.sort_values('codlocal')

Number of unique values of codlocal is 52,870
Number of records is 52,870


##### 4.4.2.6 2022

In [79]:
file = 'data/Intervenciones/Anexo 4_UGM.xlsx'
df_0 = pd.read_excel(file, sheet_name='Mto preventivo 2022-I')

In [80]:
df = df_0.copy()

# Seleccionando información
df.columns = df.iloc[1, :]
df = df[2:].reset_index(drop=True)

# Seleccionando variables
codlocal(df, 'Código del local educativo', 'codlocal')
df.rename({'Monto incurrido en el mantenimiento\n(S/)': 'monto_mp'},
          axis=1, inplace=True)
df = df[['codlocal', 'monto_mp']]

# Solo LL.EE. con monto
df = df[df['monto_mp'] > 0]
df['monto_mp'] = df['monto_mp'].astype(float)
df.drop_duplicates(inplace=True)
unique(df, ['codlocal'])
mp_2022 = df.copy()

print(df['monto_mp'].isna().sum())
# df.sort_values('codlocal')

Number of unique values of codlocal is 52,937
Number of records is 52,937
0


##### 4.4.2.7 2023

###### 2023-0

In [81]:
file = 'data/Intervenciones/Anexo 4_UGM.xlsx'
df_0 = pd.read_excel(file, sheet_name='Mto preventivo 2023-0')

In [82]:
df = df_0.copy()

# Seleccionando información
df.columns = df.iloc[1, :]
df = df[2:].reset_index(drop=True)

# Seleccionando variables
codlocal(df, 'Código del local educativo', 'codlocal')
df.rename({'Monto incurrido en el mantenimiento\n(S/)': 'monto_mp'},
          axis=1, inplace=True)
df = df[['codlocal', 'monto_mp']]

# Solo LL.EE. con monto
df = df[df['monto_mp'] > 0]
df['monto_mp'] = df['monto_mp'].astype(float)
df.drop_duplicates(inplace=True)
unique(df, ['codlocal'])
mp_2023_0 = df.copy()

# print(df['monto_mp'].isna().sum())

Number of unique values of codlocal is 17,160
Number of records is 17,160


###### 2023-1

In [83]:
file = 'data/Intervenciones/Anexo 4_UGM.xlsx'
df_0 = pd.read_excel(file, sheet_name='Mto preventivo 2023-I')

In [84]:
df = df_0.copy()

# Seleccionando información
df.columns = df.iloc[1, :]
df = df[2:].reset_index(drop=True)

# Seleccionando variables
codlocal(df, 'Código del local educativo', 'codlocal')
df.rename({'Monto incurrido en el mantenimiento\n(S/)': 'monto_mp'},
          axis=1, inplace=True)
df = df[['codlocal', 'monto_mp']]

# Solo LL.EE. con monto
df = df[df['monto_mp'] > 0]
df['monto_mp'] = df['monto_mp'].astype(float)
df.drop_duplicates(inplace=True)
unique(df, ['codlocal'])
mp_2023_1 = df.copy()

# print(df['monto_mp'].isna().sum())

Number of unique values of codlocal is 53,152
Number of records is 53,152


###### 2023-3

In [85]:
file = 'data/Intervenciones/Anexo 4_UGM.xlsx'
df_0 = pd.read_excel(file, sheet_name='Mto preventivo 2023-3')

In [86]:
df = df_0.copy()

# Seleccionando información
df.columns = df.iloc[1, :]
df = df[2:].reset_index(drop=True)

# Seleccionando variables
codlocal(df, 'Código del local educativo', 'codlocal')
df.rename({'Monto incurrido en el mantenimiento\n(S/)': 'monto_mp'},
          axis=1, inplace=True)
df = df[['codlocal', 'monto_mp']]

# Solo LL.EE. con monto
df = df[df['monto_mp'] > 0]
df['monto_mp'] = df['monto_mp'].astype(float)
df.drop_duplicates(inplace=True)
unique(df, ['codlocal'])
mp_2023_3 = df.copy()

# print(df['monto_mp'].isna().sum())

Number of unique values of codlocal is 4,197
Number of records is 4,197


###### Consolidacion

In [87]:
mp_2023 = pd.concat([mp_2023_0, mp_2023_1, mp_2023_3],
                    ignore_index=True)
mp_2023 = mp_2023.groupby('codlocal', as_index=False).sum()
unique(mp_2023, ['codlocal'])

Number of unique values of codlocal is 53,409
Number of records is 53,409


##### 4.4.2.8 2024

In [88]:
file = 'data/Intervenciones/Anexo 4_UGM.xlsx'
df_0 = pd.read_excel(file, sheet_name='Mto preventivo 2024-I')

In [89]:
df = df_0.copy()

# Seleccionando información
df.columns = df.iloc[1, :]
df = df[2:].reset_index(drop=True)

# Seleccionando variables
codlocal(df, 'Código del local educativo', 'codlocal')
df.rename({'Monto incurrido en el mantenimiento\n(S/)': 'monto_mp'},
          axis=1, inplace=True)
df = df[['codlocal', 'monto_mp']]

# Solo LL.EE. con monto
df = df[df['monto_mp'] > 0]
df['monto_mp'] = df['monto_mp'].astype(float)
df.drop_duplicates(inplace=True)
unique(df, ['codlocal'])
mp_2024 = df.copy()

print(df['monto_mp'].isna().sum())
# df.sort_values('codlocal')

Number of unique values of codlocal is 53,566
Number of records is 53,566
0


##### 4.4.2.9 2025

In [90]:
file = 'data/Intervenciones/Anexo 4_UGM.xlsx'
df_0 = pd.read_excel(file, sheet_name='Mto preventivo 2025-I')

In [91]:
df = df_0.copy()

# Seleccionando información
df.columns = df.iloc[1, :]
df = df[2:].reset_index(drop=True)

# Seleccionando variables
codlocal(df, 'Código del local educativo', 'codlocal')
df.rename({'Monto incurrido en el mantenimiento\n(S/)': 'monto_mp'},
          axis=1, inplace=True)
df = df[['codlocal', 'monto_mp']]

# Solo LL.EE. con monto
df = df[df['monto_mp'] > 0]
df['monto_mp'] = df['monto_mp'].astype(float)
df.drop_duplicates(inplace=True)
unique(df, ['codlocal'])
mp_2025 = df.copy()

print(df['monto_mp'].isna().sum())
# df.sort_values('codlocal')

Number of unique values of codlocal is 51,222
Number of records is 51,222
0


##### 4.4.2.10 Consolidacion

In [92]:
anios = [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
for anio in anios:
    eval(f'mp_{anio}')['anio_culm'] = anio

mp = pd.concat([mp_2017, mp_2018, mp_2019, mp_2020, mp_2021, 
                mp_2022, mp_2023, mp_2024, mp_2025],
               ignore_index=True)
MONTO_TOTAL = mp['monto_mp'].sum()
print(f'Monto total transferido y gastado: S/ {MONTO_TOTAL:,.2f}')

# Exportando data
freq(mp, 'anio_culm')
unique(mp, ['codlocal', 'anio_culm'])
mp.to_csv('data/procesadas/pronied_ugm_mto_preventivo.csv', index=False)

Monto total transferido y gastado: S/ 3,174,550,073.36
              Freq    (%)
anio_culm                
2024        53,566  12.7%
2023        53,409  12.6%
2022        52,937  12.5%
2021        52,870  12.5%
2025        51,222  12.1%
2020        49,830  11.8%
2019        48,881  11.6%
2018        34,540   8.2%
2017        25,463   6.0%
TOTAL      422,718   100% 

Number of unique values of codlocal anio_culm is 422,718
Number of records is 422,718


#### 4.4.3 Mantenimiento correctivo (pendiente)

### 4.5 UGSC

#### 4.5.1 ASITEC / SIAT

In [94]:
file1 = 'data/Intervenciones/'
file2 = 'Anexo 05 _UGSC.xlsx'
df_0 = pd.read_excel(file1 + file2, sheet_name='ASITEC_SIAT (edited)')

In [95]:
df = df_0.copy()

# Seleccionando información
df.rename({'Código Único de Inversión (CUI)': 'CUI'}, axis=1, inplace=True)
df['CUI'] = df['CUI'].astype('string')
codlocal(df, 'Código del local educativo', 'codlocal')

# Criterios para la selección de intervenciones
# i. Año de culminación hasta 2025
df['fec_culm'] = pd.to_datetime(df['Fecha de culminación de obra (o estimada)'], 
                                errors='coerce')
df['anio_culm'] = df['fec_culm'].dt.year.astype('Int64', errors='ignore')
df['cf1'] = np.where((df['anio_culm'].notna()) & (df['anio_culm'] <= 2025), 
                     1, 0)
freq(df, 'cf1')

# ii. Avance fisico mayor a 85%
df['cf2'] = np.where((df['Avance físico\n(%)'].notna()) & 
                     (df['Avance físico\n(%)'] >= 0.85), 1, 0)
freq(df, 'cf2')

# iii. Seleccionando intervenciones que cumplen con los criterios
df['cf'] = np.where((df['cf1'] == 1) & (df['cf2'] == 1), 1, 0)
freq(df, 'cf')
df = df.loc[df['cf'] == 1, :]

# Obteniendo información sobre LLEE nuevos y existente
exis = pd.read_csv('data/procesadas/llee_existentes.csv', 
                   dtype={'codlocal': str})
df = pd.merge(df, exis,
              on='codlocal', how='left', indicator='existente')

# Elaborando los indicadores del GI5
df['ind'] = ''
df.loc[(df['existente'] == 'both'), 'ind'] = 'GI5_1'
df.loc[(df['existente'] == 'left_only'), 'ind'] = 'GI5_2'

# Exportando data
freq(df, 'ind')
freq(df, 'anio_culm')
unique(df, ['codlocal'])
df = df[['codlocal', 'CUI', 'ind', 'anio_culm']]
df.to_csv('data/procesadas/pronied_ugsc_asitec.csv', index=False)

      Freq    (%)
cf1              
1      137  53.9%
0      117  46.1%
TOTAL  254   100% 

      Freq    (%)
cf2              
1      129  50.8%
0      125  49.2%
TOTAL  254   100% 

      Freq    (%)
cf               
0      127  50.0%
1      127  50.0%
TOTAL  254   100% 

      Freq    (%)
ind              
GI5_1  119  93.7%
GI5_2    8   6.3%
TOTAL  127   100% 

          Freq    (%)
anio_culm            
2023        44  34.6%
2022        37  29.1%
2024        32  25.2%
2025        14  11.0%
TOTAL      127   100% 

Number of unique values of codlocal is 127
Number of records is 127


#### 4.5.2 Seguimiento y monitoreo

In [96]:
file1 = 'data/Intervenciones/Anexo 05 _UGSC.xlsx'
df_0 = pd.read_excel(file1, sheet_name='SEGUIMIENTO_MONITOREO (edited)')

In [97]:
df = df_0.copy()

# Seleccionando información
df.rename({'Código Único de Inversión (CUI)': 'CUI'}, axis=1, inplace=True)
df['CUI'] = df['CUI'].astype('string')
codlocal(df, 'Código del local educativo', 'codlocal')

# Criterios para la selección de intervenciones
# i. Proyectos que han recibido transferencias
df['Monto total transferido al 2025'].describe()
df['cf1'] = np.where((df['Monto total transferido al 2025'].notna()) & 
                     (df['Monto total transferido al 2025'] > 0), 1, 0)
freq(df, 'cf1')

# ii. Año de culminación hasta 2025
df['fec_culm'] = pd.to_datetime(df['Fecha de culminación de obra (o estimada)'], 
                                errors='coerce')
df['anio_culm'] = df['fec_culm'].dt.year.astype('Int64', errors='ignore')
df['cf2'] = np.where((df['anio_culm'].notna()) & (df['anio_culm'] <= 2025), 
                     1, 0)
freq(df, 'cf2')

# iii. Avance fisico mayor a 85%
df['cf3'] = np.where((df['Avance físico\n(%)'].notna()) & 
                     (df['Avance físico\n(%)'] >= 0.85), 1, 0)
freq(df, 'cf3')

# iv. Obras no paralizadas ni contratos resueltos
no_etapa = ['SUSPENDIDA', 'CONTRATO RESUELTO', 'PARALIZADA']
df['cf4'] = np.where(df['Etapa del PI'].isin(no_etapa), 0, 1)
freq(df, 'cf4')

# v. Seleccionando intervenciones que cumplen con los criterios
df['cf'] = np.where((df['cf1'] == 1) & (df['cf2'] == 1) & (df['cf3'] == 1) & 
                    (df['cf4'] == 1), 1, 0)
freq(df, 'cf')
df = df.loc[df['cf'] == 1, :]

# Obteniendo información sobre LLEE nuevos y existente
exis = pd.read_csv('data/procesadas/llee_existentes.csv', 
                   dtype={'codlocal': str})
df = pd.merge(df, exis,
              on='codlocal', how='left', indicator='existente')

# Elaborando los indicadores del GI5
df['ind'] = ''
df.loc[(df['existente'] == 'both'), 'ind'] = 'GI5_1'
df.loc[(df['existente'] == 'left_only'), 'ind'] = 'GI5_2'

# Exportando data
freq(df, 'ind')
freq(df, 'anio_culm')
unique(df, ['codlocal'])
df = df[['codlocal', 'CUI', 'ind', 'anio_culm']]
df.to_csv('data/procesadas/pronied_ugsc_sym.csv', index=False)

      Freq    (%)
cf1              
1      160  91.4%
0       15   8.6%
TOTAL  175   100% 

      Freq    (%)
cf2              
1      140  80.0%
0       35  20.0%
TOTAL  175   100% 

      Freq    (%)
cf3              
1      130  74.3%
0       45  25.7%
TOTAL  175   100% 

      Freq    (%)
cf4              
1      164  93.7%
0       11   6.3%
TOTAL  175   100% 

      Freq    (%)
cf               
1      126  72.0%
0       49  28.0%
TOTAL  175   100% 

      Freq    (%)
ind              
GI5_1  118  93.7%
GI5_2    8   6.3%
TOTAL  126   100% 

          Freq    (%)
anio_culm            
2023        44  34.9%
2022        38  30.2%
2024        30  23.8%
2025        14  11.1%
TOTAL      126   100% 

Number of unique values of codlocal is 126
Number of records is 126


C:\Users\Paolo\AppData\Local\Temp\ipykernel_23816\1416101528.py:16: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['fec_culm'] = pd.to_datetime(df['Fecha de culminación de obra (o estimada)'],


## 05. ANIN

### 5.1 Mantenimiento

In [98]:
file = 'data/procesadas/vinculaciones.csv'
vinc = pd.read_csv(file, dtype={'CUI': 'string', 'codlocal': 'string'})

#### 5.1.1 Enero - Agosto 2025

In [99]:
file1 = 'data/Intervenciones/ANIN - mantenimiento.xlsx'
df_0 = pd.read_excel(file1, sheet_name='MP ene-ago 2025')

In [100]:
df = df_0.copy()

# Tipo de mantenimiento. Todas las actividades son Mto preventivo
freq(df, 'Tipo de mantenimiento')

# Seleccionando información
df['CUI'] = df['CUI'].astype('string')
anin_1 = df['CUI']
anin_1 = anin_1.drop_duplicates()

                         Freq   (%)
Tipo de mantenimiento              
Mantenimiento preventivo   29  100%
TOTAL                      29  100% 



#### 5.1.2 Setiembre - Diciembre 2025

In [101]:
file1 = 'data/Intervenciones/ANIN - mantenimiento.xlsx'
df_0 = pd.read_excel(file1, sheet_name='MP set-dic 2025')

In [102]:
df = df_0.copy()

# Tipo de mantenimiento. Todas las actividades son Mto preventivo
freq(df, 'Tipo de mantenimiento')

# Seleccionando información
df['CUI'] = df['CUI'].astype('Int64').astype('string')
df = df[df['CUI'].notna()]
anin_2 = df['CUI']
anin_2 = anin_2.drop_duplicates()

                         Freq   (%)
Tipo de mantenimiento              
Mantenimiento preventivo   86  100%
TOTAL                      86  100% 



#### 5.1.3 Consolidación

In [103]:
anin = pd.concat([anin_1, anin_2]).drop_duplicates()

anin = pd.merge(anin, vinc,
                on='CUI', how='left')

# Año de culminación
anin['anio_culm'] = 2025

# Indicador: Las actividades realizadas y programadas por ANIN son suficientes
anin['ind'] = 'GI3_4'

# Exportando data
anin = anin[['codlocal', 'CUI', 'anio_culm', 'ind']]
freq(anin, 'anio_culm')
unique(anin, ['codlocal'])
anin.to_csv('data/procesadas/anin_mantenimiento.csv', index=False)

          Freq   (%)
anio_culm           
2025        28  100%
TOTAL       28  100% 

Number of unique values of codlocal is 28
Number of records is 28


### 5.2 PIRCC

In [104]:
file = 'data/procesadas/banco_inversiones_criterios.csv'
bi = pd.read_csv(file, dtype={'CODIGO_INVERSION': 'string'})

pircc_0 = pd.read_excel('data/ANIN/Cartera_PIRCC.xlsx',
                        sheet_name='Consolidado')

C:\Users\Paolo\AppData\Local\Temp\ipykernel_23816\1203410650.py:2: DtypeWarning: Columns (136) have mixed types. Specify dtype option on import or set low_memory=False.
  bi = pd.read_csv(file, dtype={'CODIGO_INVERSION': 'string'})


In [105]:
pircc = pircc_0.copy()

# Seleccionando información
pircc.columns = pircc.iloc[0, :]
pircc = pircc[1:].reset_index(drop=True)
pircc = pircc[['CODIGO LOCAL final', 'CUI', 'TIPO', 'CULMINADO', 
               'AÑO DE CULMINACIÓN', 'CONSIDERAR']]

# Seleccionando las intervenciones culminadas y a considerar
pircc = pircc[(pircc['CULMINADO'] == 'SI') & (pircc['CONSIDERAR'] == 'SI')]

# Corrigiendo código de local
codlocal(pircc, 'CODIGO LOCAL final', 'codlocal')

# Identificando año de culminación
pircc = pircc[['codlocal', 'CUI', 'TIPO', 'AÑO DE CULMINACIÓN']]
pircc = pd.merge(pircc, bi, 
                 left_on='CUI', right_on='CODIGO_INVERSION', how='left')
pircc.rename({'CUI_x': 'CUI'}, axis=1, inplace=True)
pircc.loc[(pircc['anio_culm'] == 2023) &
          (pircc['AÑO DE CULMINACIÓN'] == 'Hasta 2022'), 'anio_culm'] = 2022
pircc.loc[(pircc['anio_culm'] == 2023) &
          (pircc['AÑO DE CULMINACIÓN'] == 2022), 'anio_culm'] = 2022
pircc.loc[(pircc['anio_culm'] == 2024) &
          (pircc['AÑO DE CULMINACIÓN'] == 2022), 'anio_culm'] = 2022
pircc.loc[(pircc['anio_culm'] == 2024) &
          (pircc['AÑO DE CULMINACIÓN'] == 'Hasta 2022'), 'anio_culm'] = 2022
pircc.loc[(pircc['anio_culm'] == 2025) &
          (pircc['AÑO DE CULMINACIÓN'] == 2022), 'anio_culm'] = 2022
pircc.loc[(pircc['anio_culm'] == 2025) &
          (pircc['AÑO DE CULMINACIÓN'] == 'Hasta 2022'), 'anio_culm'] = 2022
pircc.loc[(pircc['anio_culm'].isna()) &
          (pircc['AÑO DE CULMINACIÓN'] == 2022), 'anio_culm'] = 2022
pircc.loc[(pircc['anio_culm'].isna()) &
          (pircc['AÑO DE CULMINACIÓN'] == 'Hasta 2022'), 'anio_culm'] = 2022
pircc.loc[(pircc['anio_culm'].isna()) &
          (pircc['AÑO DE CULMINACIÓN'] == '2023-2024'), 'anio_culm'] = 2024
pircc.loc[(pircc['anio_culm'].isna()) &
          (pircc['AÑO DE CULMINACIÓN'] == '2024-2025'), 'anio_culm'] = 2025
pircc = pircc[['codlocal', 'CUI', 'TIPO', 'anio_culm']]

# Intervenciones ya reportadas por PRONIED-UGRD
pronied_ugrd_mbr = pd.read_csv('data/procesadas/pronied_ugrd_mbr.csv', 
                               dtype={'FUR': 'string'})
pronied_ugrd_me = pd.read_csv('data/procesadas/pronied_ugrd_me.csv', 
                              dtype={'FUR': 'string'})
pircc = pd.merge(pircc, pronied_ugrd_mbr['FUR'], left_on='CUI', right_on='FUR', 
                 how='left', indicator='_mbr')
pircc = pd.merge(pircc, pronied_ugrd_me['FUR'], left_on='CUI', right_on='FUR', 
                 how='left', indicator='_me')
pircc['considerado'] = np.where((pircc['_mbr'] == 'left_only') &
                                (pircc['_me'] == 'left_only'), 0, 1)
print(pd.crosstab(pircc['TIPO'], pircc['considerado'], 
                  margins=True, dropna=False), '\n')
pircc = pircc[pircc['considerado'] == 0]

# Obteniendo información sobre LLEE nuevos y existente
exis = pd.read_csv('data/procesadas/llee_existentes.csv', 
                   dtype={'codlocal': str})
pircc = pd.merge(pircc, exis,
                 on='codlocal', how='left', indicator='existente')

# Elaborando los indicadores del GI5
pircc['ind'] = ''
pircc.loc[(pircc['existente'] == 'both'), 'ind'] = 'GI5_1'
pircc.loc[(pircc['existente'] == 'left_only'), 'ind'] = 'GI5_2'

# Exportando data
freq(pircc, 'anio_culm')
freq(pircc, 'ind')
unique(pircc, ['codlocal'])
pircc = pircc[['codlocal', 'CUI', 'ind', 'anio_culm']]
pircc.to_csv('data/procesadas/pircc_arcc_anin.csv', index=False)

considerado    0   1  All
TIPO                     
MBR          830   6  836
ME             0   5    5
All          830  11  841 

          Freq    (%)
anio_culm            
2,022.0    669  80.6%
2,025.0     62   7.5%
2,021.0     57   6.9%
2,024.0     40   4.8%
2,020.0      1   0.1%
2,019.0      1   0.1%
 TOTAL     830   100% 

      Freq    (%)
ind              
GI5_1  776  93.5%
GI5_2   54   6.5%
TOTAL  830   100% 

Number of unique values of codlocal is 830
Number of records is 830


## 06. DIGESE

In [106]:
file1 = 'data/Intervenciones/DIGESE.xlsx'
df_0 = pd.read_excel(file1)

In [107]:
df = df_0.copy()

# Corrección COAR Cajamarca
codlocal(df, 'Código local', 'codlocal')
df.loc[(df['codlocal'] == '394605') & 
       (df['Responsable'] == 'GORE CAJAMARCA'), 'codlocal'] = '097733'

# Supuesto 1: Solo el mantenimiento correctivo y preventivo es el adecuado
# Supuesto 2: El monto de mto. preventivo se divide entre años de periodo
freq(df, 'Tipo de mantenimiento')
mtos = ['PREVENTIVO', 'CORRECTIVO']
df = df.loc[df['Tipo de mantenimiento'].isin(mtos)]

# Año de culminación
freq(df, 'Año'); freq(df, 'Comentarios')
df['anio_culm'] = 2025 # Supuesto: Intervenciones de MTO-C son recientes
df.loc[df['Tipo de mantenimiento'] == 'PREVENTIVO', 'anio_culm'] = df['Año']
freq(df, 'anio_culm')

# Elaborando los indicadores del GI3.3
df['ind'] = np.where(df['Tipo de mantenimiento'] == 'PREVENTIVO',
                     'GI3_4', 'GI3_3')

# Exportando data
freq(df, 'ind')
freq(df, 'anio_culm')
unique(df, ['codlocal', 'anio_culm'])
df = df[['codlocal', 'ind', 'anio_culm']]
df.to_csv('data/procesadas/digese.csv', index=False)

                      Freq    (%)
Tipo de mantenimiento            
PREVENTIVO              16  72.7%
CORRECTIVO               5  22.7%
ACONDICIONAMIENTO        1   4.5%
TOTAL                   22   100% 

        Freq    (%)
Año                
 NaN       5  23.8%
2,025.0    5  23.8%
2,024.0    4  19.0%
2,023.0    2   9.5%
2,021.0    2   9.5%
2,022.0    2   9.5%
2,020.0    1   4.8%
 TOTAL    21   100% 

                        Freq    (%)
Comentarios                        
NaN                       16  76.2%
No se especificó el año    5  23.8%
TOTAL                     21   100% 

          Freq    (%)
anio_culm            
2025        10  47.6%
2024         4  19.0%
2022         2   9.5%
2021         2   9.5%
2023         2   9.5%
2020         1   4.8%
TOTAL       21   100% 

      Freq    (%)
ind              
GI3_4   16  76.2%
GI3_3    5  23.8%
TOTAL   21   100% 

          Freq    (%)
anio_culm            
2025        10  47.6%
2024         4  19.0%
2022         2   9.5%
2021    

## 07. PROINVERSION (PENDIENTE)

## 08. FONCODES

In [108]:
file = 'data/procesadas/vinculaciones.csv'
vinc = pd.read_csv(file, dtype={'CUI': 'string', 'codlocal': 'string'})

file1 = 'data/Intervenciones/FONCODES.xlsx'
df_0 = pd.read_excel(file1)

In [109]:
df = df_0.copy()

# Seleccionando información
df.columns = df.iloc[1, :]
df = df.iloc[2:-1, 1:].reset_index(drop=True)

# Solo se consideran proyectos con CUI identificados
freq(df, 'CUI')
df = df[~(df['CUI'] == 'No tiene')]

# Año de culminación
df['fec_culm'] = pd.to_datetime(df['FECHA TERMINO'], errors='coerce')
df['anio_culm'] = df['fec_culm'].dt.year.astype('Int64', errors='ignore')
freq(df, 'anio_culm')
df = df.loc[df['anio_culm'] <= 2025, :]

# Avance físico mayor al 85%
freq(df, 'AVANCE FISICO')

# Vinculaciones con los locales educativos
df = df[['CUI', 'anio_culm']]
df = pd.merge(df, vinc[['CUI', 'codlocal']],
              on='CUI', how='left')

# Indicadores: Ampliación de infraestructura
df['ind'] = 'GI4_4'

# Exportando data
freq(df, 'anio_culm')
unique(df, ['codlocal'])
df.to_csv('data/procesadas/foncodes.csv', index=False)

         Freq    (%)
CUI                 
No tiene   45  90.0%
2471659     1   2.0%
2471529     1   2.0%
2471670     1   2.0%
2471675     1   2.0%
2471685     1   2.0%
TOTAL      50   100% 

          Freq   (%)
anio_culm           
2024         5  100%
TOTAL        5  100% 

              Freq   (%)
AVANCE FISICO           
1                5  100%
TOTAL            5  100% 

          Freq   (%)
anio_culm           
2024         5  100%
TOTAL        5  100% 

Number of unique values of codlocal is 5
Number of records is 5


## 09. DRELM (PENDIENTE)

## 10. Otras UE del GN

In [110]:
peip = pd.read_csv('data/procesadas/peip_intervenciones.csv', 
                   dtype={'CUI': 'string'})

pronied_ugeo = pd.read_csv('data/procesadas/pronied_ugeo_pre.csv', 
                           dtype={'CUI': 'string'})

pronied_ugrd_mbr = pd.read_csv('data/procesadas/pronied_ugrd_mbr.csv', 
                               dtype={'FUR': 'string'})

pronied_ugrd_me = pd.read_csv('data/procesadas/pronied_ugrd_me.csv', 
                              dtype={'FUR': 'string'})

pronied_ugme_me = pd.read_csv('data/procesadas/pronied_ugme_me_prev.csv', 
                              dtype={'FUR': 'string'})

pronied_ugsc_asitec = pd.read_csv('data/procesadas/pronied_ugsc_asitec.csv', 
                                  dtype={'CUI': 'string'})

pronied_ugsc_sym = pd.read_csv('data/procesadas/pronied_ugsc_sym.csv', 
                               dtype={'CUI': 'string'})

pircc_arcc_anin = pd.read_csv('data/procesadas/pircc_arcc_anin.csv', 
                              dtype={'CUI': 'string'})

file = 'data/procesadas/vinculaciones.csv'
vinc = pd.read_csv(file, dtype={'CUI': 'string', 'codlocal': 'string'})
vinc = vinc[~((vinc['CUI'] == '2040269') & (vinc['codlocal'] == '329667'))]
vinc = vinc[~((vinc['CUI'] == '2040272') & (vinc['codlocal'] == '329667'))]
vinc = vinc[~((vinc['CUI'] == '2040273') & (vinc['codlocal'] == '329667'))]
vinc = vinc[~(vinc['CUI'] == '2488226')]

In [111]:
df_0 = pd.read_csv('data/procesadas/banco_inversiones_gn.csv', 
                   dtype={'CUI': str})
freq(df_0, 'DES_TIPO_FORMATO')
unique(df_0, ['CUI'])

                      Freq    (%)
DES_TIPO_FORMATO                 
PROYECTO DE INVERSION  477  74.4%
PROYECTO EMBLEMATICO    90  14.0%
IOARR                   42   6.6%
FUR                     32   5.0%
TOTAL                  641   100% 

Number of unique values of CUI is 641
Number of records is 641


### 10.1 FUR y proyectos emblemáticos

In [112]:
df = df_0.copy()

# No existe descripción detallada de la intervención
tipos = ['FUR', 'PROYECTO EMBLEMATICO']
df.loc[df['DES_TIPO_FORMATO'].isin(tipos), 'ALTERNATIVA'].unique()

# Identificación de intervenciones de unidades ejecutoras del GN
df = pd.merge(df, peip['CUI'], on='CUI', how='left', indicator='_peip')
df.drop_duplicates(inplace=True)
df = pd.merge(df, pronied_ugeo['CUI'], on='CUI', how='left', indicator='_ugeo')
df = pd.merge(df, pronied_ugrd_mbr['FUR'], left_on='CUI', right_on='FUR', 
              how='left', indicator='_mbr')
df = pd.merge(df, pronied_ugrd_me['FUR'], left_on='CUI', right_on='FUR', 
              how='left', indicator='_me')
# df = pd.merge(df, pronied_ugme_me['CUI'].drop_duplicates(), on='CUI', 
#               how='left', indicator='_ugme_me')
df = pd.merge(df, pronied_ugsc_asitec['CUI'], on='CUI', how='left', 
              indicator='_asitec')
df = pd.merge(df, pronied_ugsc_sym['CUI'], on='CUI', how='left', 
              indicator='_sym')
df = pd.merge(df, pircc_arcc_anin['CUI'], on='CUI', how='left', 
              indicator='_pircc')

df['considerado'] = np.where((df['_peip'] == 'left_only') & 
                             (df['_ugeo'] == 'left_only') &
                             (df['_mbr'] == 'left_only') &
                             (df['_me'] == 'left_only') &
                        #    (df['_ugme_me'] == 'left_only') &
                             (df['_asitec'] == 'left_only') &
                             (df['_sym'] == 'left_only') &
                             (df['_pircc'] == 'left_only'), 0, 1)
freq(df[df['DES_TIPO_FORMATO'].isin(tipos)], 'considerado')
freq(df[(df['DES_TIPO_FORMATO'].isin(tipos)) & (df['considerado'] == 0)], 
     'DES_TIPO_FORMATO')

            Freq    (%)
considerado            
0             69  56.6%
1             53  43.4%
TOTAL        122   100% 

                     Freq    (%)
DES_TIPO_FORMATO                
PROYECTO EMBLEMATICO   46  66.7%
FUR                    23  33.3%
TOTAL                  69   100% 



#### 10.1.1 Vinculacion manual de los FUR no considerados

In [113]:
fur_0 = pd.read_excel('data/procesadas/FUR_validar_Melissa.xlsx',
                      dtype={'CUI': str}, sheet_name='Data')

In [114]:
fur = fur_0.copy()

# Seleccionando variables
fur = fur[['CUI', 
           'GI1.1', 'GI1.2', 'GI1.3', 'GI1.4', 'GI1.5', 
           'GI2.1', 'GI2.2', 
           'GI3.1', 'GI3.2', 'GI3.3', 'GI3.4', 
           'GI4.1', 'GI4.2', 'GI4.3', 'GI4.4', 
           'GI5']]
fur.iloc[:, 1:] = fur.iloc[:, 1:].fillna(0)

cols = ['GI1.1', 'GI1.2', 'GI1.3', 'GI1.4', 'GI1.5', 
        'GI2.1', 'GI2.2', 
        'GI3.1', 'GI3.2', 'GI3.3', 'GI3.4', 
        'GI4.1', 'GI4.2', 'GI4.3', 'GI4.4', 
        'GI5']
sufix = '_fur_vinculados'
fur = fur.rename(columns={
    c: c.replace('.', '_') + sufix 
    for c in cols if c in fur.columns
})

unique(fur, ['CUI'])

Number of unique values of CUI is 23
Number of records is 23


In [115]:
# Supuesto 1: EMBLEMATICOS implican la sustitución total de la infraestructura
# Supuesto 2: Los FUR no necesariamente, se vinculó manualmente
aux = df[(df['DES_TIPO_FORMATO'].isin(tipos)) & (df['considerado'] == 0)]
aux = pd.merge(aux, fur, on='CUI', how='left')

# Obteniendo información sobre LLEE nuevos y existente
exis = pd.read_csv('data/procesadas/llee_existentes.csv', 
                   dtype={'codlocal': str})
aux = pd.merge(aux, vinc[['CUI', 'codlocal']],
               on='CUI', how='left', indicator='CUI_codlocal')
aux = pd.merge(aux, exis,
               on='codlocal', how='left', indicator='existente')

# Elaborando los indicadores del GI5
aux = aux[['CUI', 'existente', 'DES_TIPO_FORMATO',
           'GI1_1_fur_vinculados', 'GI1_2_fur_vinculados', 
           'GI1_3_fur_vinculados', 'GI1_4_fur_vinculados', 
           'GI1_5_fur_vinculados', 'GI2_1_fur_vinculados',
           'GI2_2_fur_vinculados', 'GI3_1_fur_vinculados', 
           'GI3_2_fur_vinculados', 'GI3_3_fur_vinculados', 
           'GI3_4_fur_vinculados', 'GI4_1_fur_vinculados',
           'GI4_2_fur_vinculados', 'GI4_3_fur_vinculados', 
           'GI4_4_fur_vinculados', 'GI5_fur_vinculados']]

aux['GI5_1_otros'] = 0
aux.loc[(aux['existente'] == 'both') & 
        (aux['DES_TIPO_FORMATO'] == 'PROYECTO EMBLEMATICO'), 'GI5_1_otros'] = 1
aux.loc[(aux['existente'] == 'both') & 
        (aux['DES_TIPO_FORMATO'] == 'FUR') &
        (aux['GI5_fur_vinculados'] == 1), 'GI5_1_otros'] = 1

aux['GI5_2_otros'] = 0
aux.loc[(aux['existente'] == 'left_only') & 
        (aux['DES_TIPO_FORMATO'] == 'PROYECTO EMBLEMATICO'), 'GI5_2_otros'] = 1
aux.loc[(aux['existente'] == 'left_only') & 
        (aux['DES_TIPO_FORMATO'] == 'FUR') &
        (aux['GI5_fur_vinculados'] == 1), 'GI5_2_otros'] = 1
aux.drop_duplicates(inplace=True)

# Dimensionando a nivel de CUI
aux = aux.groupby(['CUI']) \
    [['GI1_1_fur_vinculados', 'GI1_2_fur_vinculados', 'GI1_3_fur_vinculados', 
      'GI1_4_fur_vinculados', 'GI1_5_fur_vinculados', 'GI2_1_fur_vinculados',
      'GI2_2_fur_vinculados', 'GI3_1_fur_vinculados', 'GI3_2_fur_vinculados', 
      'GI3_3_fur_vinculados', 'GI3_4_fur_vinculados', 'GI4_1_fur_vinculados',
      'GI4_2_fur_vinculados', 'GI4_3_fur_vinculados', 'GI4_4_fur_vinculados',
      'GI5_1_otros', 'GI5_2_otros']].sum().reset_index()
aux = aux.rename(columns={
    c: c.replace('fur_vinculados', 'otros') for c in aux.columns
})

aux.to_csv('data/procesadas/FUR_emblematicos.csv', index=False)

### 10.2 Proyecto de inversión

In [116]:
pi = df.loc[df['DES_TIPO_FORMATO'] == 'PROYECTO DE INVERSION']
freq(pi, 'considerado')
freq(pi[pi['considerado'] == 0], 'PLIEGO')

pi = pi[pi['considerado'] == 0]

            Freq    (%)
considerado            
0            367  76.9%
1            110  23.1%
TOTAL        477   100% 

                                                   Freq    (%)
PLIEGO                                                        
M. DE EDUCACION                                     192  52.3%
MINISTERIO DE LA MUJER Y DESARROLLO SOCIAL          153  41.7%
INSTITUTO NACIONAL DE INFRAESTRUCTURA EDUCATIVA...    6   1.6%
M. DE DEFENSA                                         4   1.1%
U.N. HERMILIO VALDIZAN                                3   0.8%
U.N. DE PIURA                                         2   0.5%
M. DEL INTERIOR                                       2   0.5%
M. DE DESARROLLO AGRARIO Y RIEGO                      2   0.5%
U.N. DE TRUJILLO                                      1   0.3%
SERVICIO NACIONAL DE CAPACITACION PARA LA INDUS...    1   0.3%
U.N. MAYOR DE SAN MARCOS                              1   0.3%
TOTAL                                               367   1

#### 10.2.1 Grupo de intervención 1

##### GI1.1

In [117]:
grupos = [
    ['demolicion', 'demoler'],
    ['provisional', 'provisionales', 'prefabricada', 'prefabricadas',
     'modular', 'modulares', 'temporal', 'temporales']
]

pi = flag_word_groups_in_columns(
    pi, 
    cols=['NOMBRE_INVERSION', 'ALTERNATIVA'],
    groups=grupos,
    new_col='GI1_1_pre',
    whole_word=True
)

freq(pi, 'GI1_1_pre')

          Freq    (%)
GI1_1_pre            
0          352  95.9%
1           15   4.1%
TOTAL      367   100% 



##### GI1.2

In [118]:
grupos = [
    ['reforzamiento', 'reforzar']
]

#prohibidas = ['demolicion', 'construccion', 'construir']

pi = flag_word_groups_in_columns(
    pi, 
    cols=['NOMBRE_INVERSION', 'ALTERNATIVA'],
    groups=grupos,
#    forbidden=prohibidas,
    new_col='GI1_2_pre',
    whole_word=True
)

# Se identificaron manualmente los siguientes CUIs:
reforza = ['2088275', '2057442', '2088291', '2056319', '2088274', '2057444', 
           '2078562', '2078567', '2234735', '2088283', '2058138', '2057441', 
           '2078568', '2057486']
pi.loc[pi['CUI'].isin(reforza), 'GI1_2_pre'] = 1
freq(pi, 'GI1_2_pre')

          Freq    (%)
GI1_2_pre            
0          348  94.8%
1           19   5.2%
TOTAL      367   100% 



##### GI1.3

In [119]:
grupos = [
    ['intervencion'],
    ['contingente'],
    ['geotextil']
]

pi = flag_word_groups_in_columns(
    pi, 
    cols=['NOMBRE_INVERSION', 'ALTERNATIVA'],
    groups=grupos,
    new_col='GI1_3_pre',
    whole_word=True
)

freq(pi, 'GI1_3_pre')

          Freq   (%)
GI1_3_pre           
0          367  100%
TOTAL      367  100% 



##### GI1.4

In [120]:
grupos = [
    ['modular', 'modulares'],
    ['selva', 'amazonia', 'amazonico', 'tropical']
]

pi = flag_word_groups_in_columns(
    pi, 
    cols=['NOMBRE_INVERSION', 'ALTERNATIVA'],
    groups=grupos,
    new_col='GI1_4_pre',
    whole_word=True
)

freq(pi, 'GI1_4_pre')

          Freq   (%)
GI1_4_pre           
0          367  100%
TOTAL      367  100% 



##### GI1.5

In [121]:
grupos = [
    ['cerco', 'muro'],
    ['perimetrico']
]

pi = flag_word_groups_in_columns(
    pi, 
    cols=['NOMBRE_INVERSION', 'ALTERNATIVA'],
    groups=grupos,
    new_col='GI1_5_pre',
    whole_word=True
)

# Se identificaron manualmente los siguientes CUIs:
cercos = ['2030239', '2030234']
pi.loc[pi['CUI'].isin(cercos), 'GI1_5_pre'] = 1

freq(pi, 'GI1_5_pre')

          Freq    (%)
GI1_5_pre            
0          233  63.5%
1          134  36.5%
TOTAL      367   100% 



#### 10.2.2 Grupo de intervención 2

##### GI2.1

In [122]:
grupos = [
    ['agua', 'potable', 'desague', 'desagüe', 'alcantarillado', 'saneamiento']
]

pi = flag_word_groups_in_columns(
    pi, 
    cols=['NOMBRE_INVERSION', 'ALTERNATIVA'],
    groups=grupos,
    new_col='GI2_1_pre',
    whole_word=True
)

freq(pi, 'GI2_1_pre')

          Freq    (%)
GI2_1_pre            
0          340  92.6%
1           27   7.4%
TOTAL      367   100% 



##### GI2.2

In [123]:
grupos = [
    ['almacenamiento', 'cisterna', 'tanque', 'elevado', 'bombeo', 'bomba',
     'higienicos', 'ss.hh.', 'sshh', 'inodoro', 'urinario', 'lavadero',
     'bebedero', 'bebederos', 'drenaje', 'lluvia', 'lluvia', 'pluvial', 
     'canaleta', 'canaletas']
]

pi = flag_word_groups_in_columns(
    pi, 
    cols=['NOMBRE_INVERSION', 'ALTERNATIVA'],
    groups=grupos,
    new_col='GI2_2_pre',
    whole_word=True
)

freq(pi, 'GI2_2_pre')

          Freq    (%)
GI2_2_pre            
0          200  54.5%
1          167  45.5%
TOTAL      367   100% 



#### 10.2.3 Grupo de intervención 3

##### GI3.1

In [124]:
grupos = [
    ['cable', 'cableado', 'tablero', 'tableros', 'gabinete', 'gabinetes',
     'interruptores', 'interruptor', 'puesta a tierra', 
     'instalaciones internas']
]

pi = flag_word_groups_in_columns(
    pi, 
    cols=['NOMBRE_INVERSION', 'ALTERNATIVA'],
    groups=grupos,
    new_col='GI3_1_pre',
    whole_word=True
)

freq(pi, 'GI3_1_pre')

          Freq    (%)
GI3_1_pre            
0          364  99.2%
1            3   0.8%
TOTAL      367   100% 



##### GI3.2

In [125]:
grupos = [
    ['mobiliario', 'muebles', 'carpeta', 'silla', 'escritorio', 'pizarra',
     'mesa', 'sillon', 'corral'], 
    ['equipamiento', 'equipos']
]

pi = flag_word_groups_in_columns(
    pi, 
    cols=['NOMBRE_INVERSION', 'ALTERNATIVA'],
    groups=grupos,
    new_col='GI3_2_pre',
    whole_word=False
)

freq(pi, 'GI3_2_pre')

          Freq    (%)
GI3_2_pre            
0          189  51.5%
1          178  48.5%
TOTAL      367   100% 



#### 10.2.4 Grupo de intervención 4

##### GI4.1

Supuesto: La sustitución parcial se da mediante IOARRs. Solo se identificaron
los siguientes proyectos.

In [126]:
parcial = ['2088275', '2057442', '2088291', '2056319', '2088274', '2057444', 
           '2078562', '2078567', '2234735', '2088283', '2058138', '2057441', 
           '2078568', '2057486']
pi['GI4_1_pre'] = np.where(pi['CUI'].isin(parcial), 1, 0)
freq(pi, 'GI4_1_pre')

          Freq    (%)
GI4_1_pre            
0          353  96.2%
1           14   3.8%
TOTAL      367   100% 



##### GI4.2

In [127]:
grupos = [
    ['electrica', 'energia', 'electricidad', 'eolica', 'solar']
]

pi = flag_word_groups_in_columns(
    pi, 
    cols=['NOMBRE_INVERSION', 'ALTERNATIVA'],
    groups=grupos,
    new_col='GI4_2_pre',
    whole_word=True
)

freq(pi, 'GI4_2_pre')

          Freq    (%)
GI4_2_pre            
0          360  98.1%
1            7   1.9%
TOTAL      367   100% 



##### GI4.3

In [128]:
grupos = [
    ['rampa', 'ascensor', 'elevador', 'inodoro accesible', 'discapacidad', 
     "instalaciones sanitarias accesibles", "aparatos sanitarios", 
     "sshh accesibles", "inodoro accesible", "bano accesible", 
     "bateria sanitaria accesible", "lavatorio accesible", 
     "personas con discapacidad", "discapacidad"]
]

pi = flag_word_groups_in_columns(
    pi, 
    cols=['NOMBRE_INVERSION', 'ALTERNATIVA'],
    groups=grupos,
    new_col='GI4_3_pre',
    whole_word=True
)

freq(pi, 'GI4_3_pre')

          Freq    (%)
GI4_3_pre            
0          362  98.6%
1            5   1.4%
TOTAL      367   100% 



##### GI4.4

In [129]:
grupos = [
    ["ampliacion", "ampliar", "ampliada", "incremento de area", 
     "nuevos espacios"],
    ["infraestructura", "local educativo", "edificacion educativa", 
     "espacios educativos"]
]

pi = flag_word_groups_in_columns(
    pi, 
    cols=['NOMBRE_INVERSION', 'ALTERNATIVA'],
    groups=grupos,
    new_col='GI4_4_pre',
    whole_word=True
)

freq(pi, 'GI4_4_pre')

          Freq    (%)
GI4_4_pre            
0          278  75.7%
1           89  24.3%
TOTAL      367   100% 



#### 10.2.5 Grupo de intervención 5

In [130]:
grupos = [
    ['sustitucion', 'sustituir', 'construccion'],
    ['edificacion', 'pabellon', 'aula', 'infraestructura'],
]

pi = flag_word_groups_in_columns(
    pi, 
    cols=['NOMBRE_INVERSION', 'ALTERNATIVA'],
    groups=grupos,
    new_col='sust_total_nueva_infra',
    whole_word=False
)

freq(pi, 'sust_total_nueva_infra')

                       Freq    (%)
sust_total_nueva_infra            
1                       309  84.2%
0                        58  15.8%
TOTAL                   367   100% 



In [131]:
# Obteniendo información sobre LLEE nuevos y existente
exis = pd.read_csv('data/procesadas/llee_existentes.csv', 
                   dtype={'codlocal': str})
# vinc = pd.read_csv('data/procesadas/vinculaciones.csv',
#                    dtype={'codlocal': str, 'CUI': str})
# vinc = vinc[~((vinc['CUI'] == '2040269') & (vinc['codlocal'] == '329667'))]
# vinc = vinc[~((vinc['CUI'] == '2040272') & (vinc['codlocal'] == '329667'))]
# vinc = vinc[~((vinc['CUI'] == '2040273') & (vinc['codlocal'] == '329667'))]
# vinc = vinc[~(vinc['CUI'] == '2488226')]

aux = pd.merge(pi, vinc[['CUI', 'codlocal']],
               on='CUI', how='left', indicator='CUI_codlocal')
aux = pd.merge(aux, exis,
               on='codlocal', how='left', indicator='existente')

# Elaborando los indicadores del GI5
aux = aux[['CUI', 'sust_total_nueva_infra', 'existente']]
aux.drop_duplicates(inplace=True)

aux['GI5_1_pre'] = 0
aux.loc[(aux['sust_total_nueva_infra'] == 1) & (aux['existente'] == 'both'),
       'GI5_1_pre'] = 1

aux['GI5_2_pre'] = 0
aux.loc[(aux['sust_total_nueva_infra'] == 1) & 
        (aux['existente'] == 'left_only'), 'GI5_2_pre'] = 1

# Dimensionando a nivel de CUI
aux = aux.groupby('CUI')[['GI5_1_pre', 'GI5_2_pre']].sum().reset_index()
freq(aux, 'GI5_1_pre')
freq(aux, 'GI5_2_pre')

# Consolidando
pi = pd.merge(pi, aux, on='CUI', how='left')

          Freq    (%)
GI5_1_pre            
1          303  82.6%
0           64  17.4%
TOTAL      367   100% 

          Freq    (%)
GI5_2_pre            
0          359  97.8%
1            8   2.2%
TOTAL      367   100% 



#### 10.2.6 Cambios extras

In [132]:
# UE 113, UE 120 y SENCICO no tienen intervencion en infraestructura educativa
ind_pnie = ['GI1_1_pre', 'GI1_2_pre', 'GI1_3_pre', 'GI1_4_pre', 'GI1_5_pre', 
            'GI2_1_pre', 'GI2_2_pre', 
            'GI3_1_pre', 'GI3_2_pre', 
            'GI4_1_pre', 'GI4_2_pre', 'GI4_3_pre', 'GI4_4_pre',
            'GI5_1_pre', 'GI5_2_pre']
for ind in ind_pnie:
    pi.loc[pi['COD_EJECUTORA'] == 113, ind] = 0
    pi.loc[pi['COD_EJECUTORA'] == 120, ind] = 0
    pi.loc[(pi['COD_EJECUTORA'] == 1) & 
           (pi['UF'] == \
            'OFICINA DE DESARROLLO MANTENIMIENTO E INFRAESTRUCTURA'), ind] = 0

# UE 125 - PEIP es sustitucion total
pi['sust_total'] = np.where(pi['COD_EJECUTORA'] == 125, 1, 0)

# UE 108 - PRONIED es sustitucion total excepto algunos CUIs
no_sust_total = ['2030239', '2030234', '2088275', '2057442', '2088291', 
                 '2056319', '2088274', '2057444', '2078562', '2078567', 
                 '2234735', '2088283', '2058138', '2057441', '2078568', 
                 '2057486']
pi.loc[pi['COD_EJECUTORA'] == 108, 'sust_total'] = 1
pi.loc[pi['CUI'].isin(no_sust_total), 'sust_total'] = 0

# Intervenciones del INFE es sustitucion total
pi.loc[(pi['COD_EJECUTORA'] == 1) &
       (pi['UF'] ==
        'INSTITUTO NACIONAL DE INFRAESTRUCTURA EDUCATIVA Y DE SALUD-INFES'), 
        'sust_total'] = 1

# Obteniendo información sobre LLEE nuevos y existente
aux = pi.loc[pi['sust_total'] == 1]
aux = pd.merge(aux, vinc[['CUI', 'codlocal']],
               on='CUI', how='left', indicator='CUI_codlocal')
aux = pd.merge(aux, exis,
               on='codlocal', how='left', indicator='existente')

# Elaborando los indicadores del GI5
aux = aux[['CUI', 'sust_total', 'existente']]
aux.drop_duplicates(inplace=True)

aux['GI5_1_pre_extra'] = 0
aux.loc[(aux['existente'] == 'both'), 'GI5_1_pre_extra'] = 1

aux['GI5_2_pre_extra'] = 0
aux.loc[(aux['existente'] == 'left_only'), 'GI5_2_pre_extra'] = 1

# Dimensionando a nivel de CUI
aux = aux.groupby('CUI')[['GI5_1_pre_extra', 'GI5_2_pre_extra']].sum(). \
    reset_index()
pi = pd.merge(pi, aux, on='CUI', how='left', indicator='_sust_total')
pi.loc[pi['_sust_total'] == 'both', 'GI5_1_pre'] = pi['GI5_1_pre_extra']
pi.loc[pi['_sust_total'] == 'both', 'GI5_2_pre'] = pi['GI5_2_pre_extra']
pi.drop(columns=['sust_total', 'GI5_1_pre_extra', 'GI5_2_pre_extra', 
                 '_sust_total'], inplace=True)

# Exportando data
pi = pi[['CUI',
         'GI1_1_pre', 'GI1_2_pre', 'GI1_3_pre', 'GI1_4_pre', 'GI1_5_pre', 
         'GI2_1_pre', 'GI2_2_pre', 
         'GI3_1_pre', 'GI3_2_pre',
         'GI4_1_pre', 'GI4_2_pre', 'GI4_3_pre', 'GI4_4_pre',
         'GI5_1_pre', 'GI5_2_pre']]
unique(pi, ['CUI'])
pi.to_csv('data/procesadas/pi_gn_final.csv', index=False)

Number of unique values of CUI is 367
Number of records is 367


### 10.3 Consolidando

In [133]:
# Considerando solo las intervenciones no consideradas por las otras UEs
freq(df, 'considerado')
df2 = df[df['considerado'] == 0]

# Seleccionando variables
df2 = df2[['CUI', 'anio_culm', 'DES_TIPO_FORMATO']]

# Obteniendo los indicadores de los FUR, emblematicos, PI e IOARR
freq(df2, 'DES_TIPO_FORMATO')
f_e = pd.read_csv('data/procesadas/FUR_emblematicos.csv', dtype={'CUI': 
                                                                 'string'})
pi = pd.read_csv('data/procesadas/pi_gn_final.csv', dtype={'CUI': 'string'})
ioarr = pd.read_csv('data/procesadas/ioarr_final.csv', dtype={'CUI': 'string'})
cols = ['GI1_1', 'GI1_2', 'GI1_4', 'GI1_5', 'GI2_1', 'GI2_2', 'GI3_1', 'GI3_2', 
        'GI4_1', 'GI4_2', 'GI4_3', 'GI4_4']
sufix = '_ioarr'
ioarr = ioarr.rename(columns={
    c: c + sufix 
    for c in cols if c in ioarr.columns
})

df2 = pd.merge(df2, f_e, on='CUI', how='left', indicator='_fur_emblematicos')
df2 = pd.merge(df2, pi, on='CUI', how='left', indicator='_pi_gn')
df2 = pd.merge(df2, ioarr, on='CUI', how='left', indicator='_ioarr')

# Consolidando indicadores
df2['GI1_1'] = df2[['GI1_1_otros', 'GI1_1_pre', 'GI1_1_ioarr']]. \
    gt(0).any(axis=1).astype(int)
df2['GI1_2'] = df2[['GI1_2_otros', 'GI1_2_pre', 'GI1_2_ioarr']]. \
    gt(0).any(axis=1).astype(int)
df2['GI1_3'] = df2[['GI1_3_otros', 'GI1_3_pre']].gt(0).any(axis=1).astype(int)
df2['GI1_4'] = df2[['GI1_4_otros', 'GI1_4_pre', 'GI1_4_ioarr']]. \
    gt(0).any(axis=1).astype(int)
df2['GI1_5'] = df2[['GI1_5_otros', 'GI1_5_pre', 'GI1_5_ioarr']]. \
    gt(0).any(axis=1).astype(int)
df2['GI2_1'] = df2[['GI2_1_otros', 'GI2_1_pre', 'GI2_1_ioarr']]. \
    gt(0).any(axis=1).astype(int)
df2['GI2_2'] = df2[['GI2_2_otros', 'GI2_2_pre', 'GI2_2_ioarr']]. \
    gt(0).any(axis=1).astype(int)
df2['GI3_1'] = df2[['GI3_1_otros', 'GI3_1_pre', 'GI3_1_ioarr']]. \
    gt(0).any(axis=1).astype(int)
df2['GI3_2'] = df2[['GI3_2_otros', 'GI3_2_pre', 'GI3_2_ioarr']]. \
    gt(0).any(axis=1).astype(int)
df2['GI4_1'] = df2[['GI4_1_otros', 'GI4_1_pre', 'GI4_1_ioarr']]. \
    gt(0).any(axis=1).astype(int)
df2['GI4_2'] = df2[['GI4_2_otros', 'GI4_2_pre', 'GI4_2_ioarr']]. \
    gt(0).any(axis=1).astype(int)
df2['GI4_3'] = df2[['GI4_3_otros', 'GI4_3_pre', 'GI4_3_ioarr']]. \
    gt(0).any(axis=1).astype(int)
df2['GI4_4'] = df2[['GI4_4_otros', 'GI4_4_pre', 'GI4_4_ioarr']]. \
    gt(0).any(axis=1).astype(int)
df2['GI5_1'] = df2[['GI5_1_otros', 'GI5_1_pre']].gt(0).any(axis=1).astype(int)
df2['GI5_2'] = df2[['GI5_2_otros', 'GI5_2_pre']].gt(0).any(axis=1).astype(int)

# Fuente de información
df2['fuente'] = ''
df2.loc[df2['_fur_emblematicos'] == 'both', 'fuente'] = \
    'Análisis FUR y emblematicos GN del EV-PNIE'
df2.loc[df2['_pi_gn'] == 'both', 'fuente'] = 'Análisis BI GN del EV-PNIE'
df2.loc[df2['_ioarr'] == 'both', 'fuente'] = 'Análisis IOARR GN del EV-PNIE'
freq(df2, 'fuente')

# Exportando data
df2 = df2[['CUI', 'anio_culm',
           'GI1_1', 'GI1_2', 'GI1_3', 'GI1_4', 'GI1_5', 
           'GI2_1', 'GI2_2', 
           'GI3_1', 'GI3_2', 
           'GI4_1', 'GI4_2', 'GI4_3', 'GI4_4', 
           'GI5_1', 'GI5_2',
           'fuente']]
freq(df2, 'anio_culm')
unique(df2, ['CUI'])
df2.to_csv('data/procesadas/otros_cui_gn_final.csv', index=False)

            Freq    (%)
considerado            
0            458  71.5%
1            183  28.5%
TOTAL        641   100% 

                      Freq    (%)
DES_TIPO_FORMATO                 
PROYECTO DE INVERSION  367  80.1%
PROYECTO EMBLEMATICO    46  10.0%
FUR                     23   5.0%
IOARR                   22   4.8%
TOTAL                  458   100% 

                                           Freq    (%)
fuente                                                
Análisis BI GN del EV-PNIE                  367  80.1%
Análisis FUR y emblematicos GN del EV-PNIE   69  15.1%
Análisis IOARR GN del EV-PNIE                22   4.8%
TOTAL                                       458   100% 

          Freq    (%)
anio_culm            
2019       124  27.1%
2009        85  18.6%
2020        74  16.2%
2024        57  12.4%
2025        33   7.2%
2021        25   5.5%
2023        17   3.7%
2022        15   3.3%
2010        10   2.2%
2018        10   2.2%
2008         2   0.4%
2017         1   0.2

In [134]:
file = 'data/procesadas/banco_inversiones_criterios.csv'
bi = pd.read_csv(file, dtype={'CODIGO_INVERSION': 'string'})

df2 = pd.merge(df2, bi[['CODIGO_INVERSION', 'FUNCION', 'PROGRAMA',
                        'SUB_PROGRAMA']],
               left_on='CUI', right_on='CODIGO_INVERSION', how='left')

# Eliminando intervenciones con FUNCION en ciencia y tecnologia
df2 = df2.loc[~(df2['FUNCION'] == 'CIENCIA Y TECNOLOGÍA')]

# Eliminando intervenciones con FUNCION en cultura y deporte
df2 = df2.loc[~(df2['FUNCION'] == 'CULTURA Y DEPORTE')]
df2 = df2.loc[~(df2['FUNCION'] == 'DEPORTES')]

# Eliminando intervenciones con FUNCION en educacion y PROGRAMA en asistencia
# educativa
df2 = df2.loc[~((df2['FUNCION'] == 'EDUCACIÓN') & 
                 (df2['PROGRAMA'] == 'ASISTENCIA EDUCATIVA'))]

# Eliminando intervenciones con FUNCION en educacion y cultura y PROGRAMA en
# asistencia a educandos, capacitacion o educacion fisica y deportes
df2 = df2.loc[~((df2['FUNCION'] == 'EDUCACION Y CULTURA') & 
                 (df2['PROGRAMA'] == 'ASISTENCIA A EDUCANDOS'))]
df2 = df2.loc[~((df2['FUNCION'] == 'EDUCACION Y CULTURA') & 
                 (df2['PROGRAMA'] == 
                  'CAPACITACION Y PERFECCIONAMIENTO'))]
df2 = df2.loc[~((df2['FUNCION'] == 'EDUCACION Y CULTURA') & 
                 (df2['PROGRAMA'] == 'EDUCACION FISICA Y DEPORTES'))]

# Eliminando intervencniones en orden publico y seguridad
df2 = df2.loc[~(df2['FUNCION'] == 'ORDEN PÚBLICO Y SEGURIDAD')]

# Eliminando intervenciones en planeamiento, gestion y reserva de contingencia
df2 = df2.loc[~(df2['FUNCION'] == 
                 'PLANEAMIENTO, GESTIÓN Y RESERVA DE CONTINGENCIA')]

# Eliminando intervenciones en salud
df2 = df2.loc[~(df2['FUNCION'] == 'SALUD')]

C:\Users\Paolo\AppData\Local\Temp\ipykernel_23816\2012024438.py:2: DtypeWarning: Columns (136) have mixed types. Specify dtype option on import or set low_memory=False.
  bi = pd.read_csv(file, dtype={'CODIGO_INVERSION': 'string'})


In [135]:
# Solo se considera intervenciones identificadas en el PNIE
df2['suma'] = df2[['GI1_1', 'GI1_2', 'GI1_3', 'GI1_4', 'GI1_5', 
                   'GI2_1', 'GI2_2', 
                   'GI3_1', 'GI3_2', 
                   'GI4_1', 'GI4_2', 'GI4_3', 'GI4_4', 
                   'GI5_1', 'GI5_2']].sum(axis=1)
df2 = df2[df2['suma'] > 0].drop(columns=['suma'])

# Transformando a nivel de codlocal y año de culminación de la intervención
otros = pd.merge(df2, vinc[['CUI', 'codlocal']],
                 on='CUI', how='left', indicator='CUI_codlocal')
otros = otros[otros['CUI_codlocal'] == 'both'].drop(columns=['CUI_codlocal'])
otros = (
    otros.groupby(['codlocal', 'anio_culm'], as_index=False)
        .agg({
            'CUI': lambda x: " / ".join(sorted(x.astype(str).unique())),
            'GI1_1': 'max', 'GI1_2': 'max', 'GI1_3': 'max', 'GI1_4': 'max', 
            'GI1_5': 'max', 'GI2_1': 'max', 'GI2_2': 'max', 'GI3_1': 'max', 
            'GI3_2': 'max', 'GI4_1': 'max', 'GI4_2': 'max', 'GI4_3': 'max', 
            'GI4_4': 'max', 'GI5_1': 'max', 'GI5_2': 'max',
            'fuente': lambda x: " / ".join(sorted(x.astype(str).unique()))
        })
)

# Exportando data
freq(otros, 'anio_culm')
unique(otros, ['CUI'])
unique(otros, ['codlocal', 'anio_culm'])
otros.to_csv('data/procesadas/otros_gn_final.csv', index=False)

            Freq    (%)
anio_culm              
2023       4,182  86.4%
2019         152   3.1%
2021         146   3.0%
2009          81   1.7%
2022          80   1.7%
2020          72   1.5%
2024          59   1.2%
2025          45   0.9%
2018          13   0.3%
2010           7   0.1%
2008           2   0.0%
2014           1   0.0%
2005           1   0.0%
2004           1   0.0%
TOTAL      4,842   100% 

Number of unique values of CUI is 427
Number of records is 4,842
Number of unique values of codlocal anio_culm is 4,842
Number of records is 4,842


## 11. Consolidación de intervenciones

In [9]:
# Obteniendo información sobre LLEE nuevos y existente
exis = pd.read_csv('data/procesadas/llee_existentes.csv', 
                   dtype={'codlocal': str})

### 11.1 PEIP

In [10]:
peip = pd.read_csv('data/procesadas/peip_intervenciones.csv', 
                   dtype={'CUI': 'string', 'codlocal': 'string'})
peip = pd.merge(peip, exis,
                on='codlocal', how='left', indicator='existente')

# Elaborando los indicadores del GI5
freq(peip, 'existente')
peip['ind'] = 'GI5_1'
peip.drop(columns=['existente'], inplace=True)

# Transformando a nivel de codlocal y año de culminación de la intervención
final = (
    peip.groupby(['codlocal', 'anio_culm'])
        .agg(
            CUI=('CUI', lambda x: " / ".join(sorted(map(str, x.unique())))),
            ind_flags=('ind', lambda x: {k: 1 for k in x.unique()})
        )
        .reset_index()
)
ind_df = final['ind_flags'].apply(pd.Series).fillna(0).astype(int)
peip = pd.concat([final.drop(columns='ind_flags'), ind_df], axis=1)

# Ordenando columnas
peip['CUI'] = peip.pop('CUI')
peip.rename(columns={'CUI': 'CUI_peip'}, inplace=True)
peip['fuente_peip'] = 'PEIP-EB jul 2025'
unique(peip, ['codlocal', 'anio_culm'])

           Freq   (%)
existente            
both         45  100%
left_only     0  0.0%
right_only    0  0.0%
TOTAL        45  100% 

Number of unique values of codlocal anio_culm is 45
Number of records is 45


### 11.2 PRONIED-UGEO

In [11]:
ugeo = pd.read_csv('data/procesadas/pronied_ugeo.csv',
                   dtype={'CUI': 'string', 'codlocal': 'string'})

# Transformando a nivel de codlocal y año de culminación de la intervención
final = (
    ugeo.groupby(['codlocal', 'anio_culm'])
        .agg(
            CUI=('CUI', lambda x: " / ".join(sorted(map(str, x.unique())))),
            ind_flags=('ind', lambda x: {k: 1 for k in x.unique()})
        )
        .reset_index()
)
ind_df = final['ind_flags'].apply(pd.Series).fillna(0).astype(int)
ugeo = pd.concat([final.drop(columns='ind_flags'), ind_df], axis=1)

# Ordenando columnas
ugeo['CUI'] = ugeo.pop('CUI')
ugeo.rename(columns={'CUI': 'CUI_ugeo'}, inplace=True)
ugeo['fuente_ugeo'] = 'PRONIED-UGEO'
unique(ugeo, ['codlocal', 'anio_culm'])

Number of unique values of codlocal anio_culm is 166
Number of records is 166


### 11.3 PRONIED-UGRD-MBR

In [12]:
ugrd_mbr = pd.read_csv('data/procesadas/pronied_ugrd_mbr.csv',
                       dtype={'FUR': 'string', 'codlocal': 'string'})

# Transformando a nivel de codlocal y año de culminación de la intervención
final = (
    ugrd_mbr.groupby(['codlocal', 'anio_culm'])
        .agg(
            CUI=('FUR', lambda x: " / ".join(sorted(map(str, x.unique())))),
            ind_flags=('ind', lambda x: {k: 1 for k in x.unique()})
        )
        .reset_index()
)
ind_df = final['ind_flags'].apply(pd.Series).fillna(0).astype(int)
ugrd_mbr = pd.concat([final.drop(columns='ind_flags'), ind_df], axis=1)

# Ordenando columnas
ugrd_mbr['CUI'] = ugrd_mbr.pop('CUI')
ugrd_mbr.rename(columns={'CUI': 'CUI_ugrd_mbr'}, inplace=True)
ugrd_mbr['fuente_ugrd_mbr'] = 'PRONIED-UGRD-MBR'
unique(ugrd_mbr, ['codlocal', 'anio_culm'])

Number of unique values of codlocal anio_culm is 55
Number of records is 55


### 11.4 PRONIED-UGRD-ME

In [13]:
ugrd_me = pd.read_csv('data/procesadas/pronied_ugrd_me.csv',
                      dtype={'FUR': 'string', 'codlocal': 'string'})

# Ordenando columnas
ugrd_me['FUR'] = ugrd_me.pop('FUR')
ugrd_me.rename(columns={'FUR': 'CUI_ugrd_me'}, inplace=True)
ugrd_me['fuente_ugrd_me'] = 'PRONIED-UGRD-ME'
unique(ugrd_me, ['codlocal', 'anio_culm'])

Number of unique values of codlocal anio_culm is 15
Number of records is 15


### 11.5 PRONIED-UGME-Sistemas Modulares

In [14]:
ugme_sm = pd.read_csv('data/procesadas/pronied_ugme_sist_modulares.csv',
                      dtype={'CUI': 'string', 'codlocal': 'string'})

# Ordenando columnas
ugme_sm['fuente_ugme_sm'] = 'PRONIED-UGME-SM'
unique(ugme_sm, ['codlocal', 'anio_culm'])

Number of unique values of codlocal anio_culm is 2,880
Number of records is 2,880


### 11.6 PRONIED-UGME-Mobiliario & Equipamiento

In [15]:
ugme_me = pd.read_csv('data/procesadas/pronied_ugme_me.csv',
                      dtype={'cui_mob_equ': 'string', 'codlocal': 'string'})

# Transformando a nivel de codlocal y año de culminación de la intervención
final = (
    ugme_me.groupby(['codlocal', 'anio_culm'])
        .agg(
            CUI=('cui_mob_equ', lambda x: " / ".join(sorted(map(str, x.unique())))),
            ind_flags=('ind', lambda x: {k: 1 for k in x.unique()})
        )
        .reset_index()
)
ind_df = final['ind_flags'].apply(pd.Series).fillna(0).astype(int)
ugme_me = pd.concat([final.drop(columns='ind_flags'), ind_df], axis=1)

# Ordenando columnas
ugme_me['CUI'] = ugme_me.pop('CUI')
ugme_me.rename(columns={'CUI': 'CUI_ugme_me'}, inplace=True)
ugme_me['fuente_ugme_me'] = 'PRONIED-UGME-ME'
unique(ugme_me, ['codlocal', 'anio_culm'])

Number of unique values of codlocal anio_culm is 1,873
Number of records is 1,873


### 11.7 PRONIED-UGM-Accesibilidad

In [16]:
ugm_acc = pd.read_csv('data/procesadas/pronied_ugm_accesibilidad.csv',
                      dtype={'CUI': 'string', 'codlocal': 'string'})

# Ordenando columnas
ugm_acc['fuente_ugm_acc'] = 'PRONIED-UGM-ACCESIBILIDAD'
unique(ugm_acc, ['codlocal', 'anio_culm'])

Number of unique values of codlocal anio_culm is 671
Number of records is 671


### 11.8 PRONIED-UGM-Mto Preventivo

In [17]:
ugm_mp = pd.read_csv('data/procesadas/pronied_ugm_mto_preventivo.csv',
                      dtype={'CUI': 'string', 'codlocal': 'string'})

# Ordenando columnas
ugm_mp['monto_mp'] = ugm_mp.pop('monto_mp')
ugm_mp.rename(columns={'monto_mp': 'GI3_4'}, inplace=True)
ugm_mp['fuente_ugm_mp'] = 'PRONIED-UGM-PREVENTIVO'
unique(ugm_mp, ['codlocal', 'anio_culm'])

Number of unique values of codlocal anio_culm is 422,718
Number of records is 422,718


### 11.9 PRONIED-UGSC-ASITEC/SIAT

In [18]:
ugsc_asitec = pd.read_csv('data/procesadas/pronied_ugsc_asitec.csv',
                      dtype={'CUI': 'string', 'codlocal': 'string'})

# Transformando a nivel de codlocal y año de culminación de la intervención
final = (
    ugsc_asitec.groupby(['codlocal', 'anio_culm'])
        .agg(
            CUI=('CUI', lambda x: " / ".join(sorted(map(str, x.unique())))),
            ind_flags=('ind', lambda x: {k: 1 for k in x.unique()})
        )
        .reset_index()
)
ind_df = final['ind_flags'].apply(pd.Series).fillna(0).astype(int)
ugsc_asitec = pd.concat([final.drop(columns='ind_flags'), ind_df], axis=1)

# Ordenando columnas
ugsc_asitec['CUI'] = ugsc_asitec.pop('CUI')
ugsc_asitec.rename(columns={'CUI': 'CUI_ugsc_asitec'}, inplace=True)
ugsc_asitec['fuente_ugsc_asitec'] = 'PRONIED-UGSC-ASITEC/SIAT'
unique(ugsc_asitec, ['codlocal', 'anio_culm'])

Number of unique values of codlocal anio_culm is 127
Number of records is 127


### 11.10 PRONIED-UGSC-Seguimiento & Monitoreo

In [19]:
ugsc_sym = pd.read_csv('data/procesadas/pronied_ugsc_sym.csv',
                      dtype={'CUI': 'string', 'codlocal': 'string'})

# Transformando a nivel de codlocal y año de culminación de la intervención
final = (
    ugsc_sym.groupby(['codlocal', 'anio_culm'])
        .agg(
            CUI=('CUI', lambda x: " / ".join(sorted(map(str, x.unique())))),
            ind_flags=('ind', lambda x: {k: 1 for k in x.unique()})
        )
        .reset_index()
)
ind_df = final['ind_flags'].apply(pd.Series).fillna(0).astype(int)
ugsc_sym = pd.concat([final.drop(columns='ind_flags'), ind_df], axis=1)

# Ordenando columnas
ugsc_sym['CUI'] = ugsc_sym.pop('CUI')
ugsc_sym.rename(columns={'CUI': 'CUI_ugsc_sym'}, inplace=True)
ugsc_sym['fuente_ugsc_sym'] = 'PRONIED-UGSC-SyM'
unique(ugsc_sym, ['codlocal', 'anio_culm'])

Number of unique values of codlocal anio_culm is 126
Number of records is 126


### 11.11 ANIN-Mantenimiento (2025)

In [20]:
anin_mto = pd.read_csv('data/procesadas/anin_mantenimiento.csv',
                      dtype={'CUI': 'string', 'codlocal': 'string'})

# Transformando a nivel de codlocal y año de culminación de la intervención
final = (
    anin_mto.groupby(['codlocal', 'anio_culm'])
        .agg(
            CUI=('CUI', lambda x: " / ".join(sorted(map(str, x.unique())))),
            ind_flags=('ind', lambda x: {k: 1 for k in x.unique()})
        )
        .reset_index()
)
ind_df = final['ind_flags'].apply(pd.Series).fillna(0).astype(int)
anin_mto = pd.concat([final.drop(columns='ind_flags'), ind_df], axis=1)

# Ordenando columnas
anin_mto['CUI'] = anin_mto.pop('CUI')
anin_mto.rename(columns={'CUI': 'CUI_anin'}, inplace=True)
anin_mto['fuente_anin_mto'] = 'ANIN'
unique(anin_mto, ['codlocal', 'anio_culm'])

Number of unique values of codlocal anio_culm is 28
Number of records is 28


### 11.12 PIRCC (ARCC/ANIN)

In [21]:
pircc = pd.read_csv('data/procesadas/pircc_arcc_anin.csv',
                    dtype={'CUI': 'string', 'codlocal': 'string'})

# Transformando a nivel de codlocal y año de culminación de la intervención
final = (
    pircc.groupby(['codlocal', 'anio_culm'])
        .agg(
            CUI=('CUI', lambda x: " / ".join(sorted(map(str, x.unique())))),
            ind_flags=('ind', lambda x: {k: 1 for k in x.unique()})
        )
        .reset_index()
)
ind_df = final['ind_flags'].apply(pd.Series).fillna(0).astype(int)
pircc = pd.concat([final.drop(columns='ind_flags'), ind_df], axis=1)

# Ordenando columnas
pircc['anio_culm'] = pircc['anio_culm'].astype(int)
pircc['CUI'] = pircc.pop('CUI')
pircc.rename(columns={'CUI': 'CUI_pircc'}, inplace=True)
pircc['fuente_pircc'] = 'ARCC / ANIN'
unique(pircc, ['codlocal', 'anio_culm'])

Number of unique values of codlocal anio_culm is 830
Number of records is 830


### 11.13 DIGESE

In [22]:
digese = pd.read_csv('data/procesadas/digese.csv',
                     dtype={'CUI': 'string', 'codlocal': 'string'})

# Transformando a nivel de codlocal y año de culminación de la intervención
final = (
    digese.groupby(['codlocal', 'anio_culm'])
        .agg(
            ind_flags=('ind', lambda x: {k: 1 for k in x.unique()})
        )
        .reset_index()
)
ind_df = final['ind_flags'].apply(pd.Series).fillna(0).astype(int)
digese = pd.concat([final.drop(columns='ind_flags'), ind_df], axis=1)

# Ordenando columnas
digese['fuente_digese'] = 'DIGESE'
unique(digese, ['codlocal', 'anio_culm'])

Number of unique values of codlocal anio_culm is 21
Number of records is 21


### 11.14 FONCODES

In [23]:
foncodes = pd.read_csv('data/procesadas/foncodes.csv',
                     dtype={'CUI': 'string', 'codlocal': 'string'})

# Transformando a nivel de codlocal y año de culminación de la intervención
final = (
    foncodes.groupby(['codlocal', 'anio_culm'])
        .agg(
            CUI=('CUI', lambda x: " / ".join(sorted(map(str, x.unique())))),
            ind_flags=('ind', lambda x: {k: 1 for k in x.unique()})
        )
        .reset_index()
)
ind_df = final['ind_flags'].apply(pd.Series).fillna(0).astype(int)
foncodes = pd.concat([final.drop(columns='ind_flags'), ind_df], axis=1)

# Ordenando columnas
foncodes['CUI'] = foncodes.pop('CUI')
foncodes.rename(columns={'CUI': 'CUI_foncodes'}, inplace=True)
foncodes['fuente_foncodes'] = 'FONCODES'
unique(foncodes, ['codlocal', 'anio_culm'])

Number of unique values of codlocal anio_culm is 5
Number of records is 5


### 11.15 Otras intervenciones del GN (PIs, FUR, emblematicos e IOARR)

In [24]:
gn = pd.read_csv('data/procesadas/otros_gn_final.csv',
                 dtype={'CUI': 'string', 'codlocal': 'string'})

# Ordenando columnas
gn['CUI'] = gn.pop('CUI')
gn.rename(columns={'CUI': 'CUI_otros_gn', 'fuente': 'fuente_otros_gn'}, 
          inplace=True)
unique(gn, ['codlocal', 'anio_culm'])

Number of unique values of codlocal anio_culm is 4,842
Number of records is 4,842


### 11.16 Otras intervenciones del GR/GL (PIs, FUR, emblematicos e IOARR)

In [25]:
gr_gl = pd.read_csv('data/procesadas/otros_gr_gl_final.csv',
                 dtype={'CUI': 'string', 'codlocal': 'string'})

# Ordenando columnas
gr_gl['CUI'] = gr_gl.pop('CUI')
gr_gl.rename(columns={'CUI': 'CUI_otros_gr_gl', 'fuente': 'fuente_otros_gr_gl'}, 
          inplace=True)
unique(gr_gl, ['codlocal', 'anio_culm'])

Number of unique values of codlocal anio_culm is 19,486
Number of records is 19,486


### 11.17 Consolidacion final

In [47]:
df = pd.concat([peip, ugeo, ugrd_mbr, ugrd_me, ugme_sm, ugme_sm, ugme_me,
                ugm_acc, ugm_mp, ugsc_asitec, ugsc_sym, anin_mto, pircc, digese,
                foncodes, gn, gr_gl], axis=0, ignore_index=True, sort=False)
df = df.fillna({'GI1_1': 0, 'GI1_2': 0, 'GI1_3': 0, 'GI1_4': 0, 'GI1_5': 0,
                'GI2_1': 0, 'GI2_2': 0, 
                'GI3_1': 0, 'GI3_2': 0, 'GI3_3': 0, 'GI3_4': 0, 
                'GI4_1': 0, 'GI4_2': 0, 'GI4_3': 0, 'GI4_4': 0, 
                'GI5_1': 0, 'GI5_2': 0})

In [ ]:
df_result = reshape_gi_cui_fuente(df)

unique(df_result, ['codlocal', 'anio_culm'])

C:\Users\Paolo\AppData\Local\Temp\ipykernel_33392\3794551682.py:50: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cui_series = g.apply(make_cui_series).rename("CUI")
C:\Users\Paolo\AppData\Local\Temp\ipykernel_33392\3794551682.py:51: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  fuente_series = g.apply(make_fuente).rename("fuente")


Number of unique values of codlocal anio_culm is 428,337
Number of records is 428,337
                                                       Freq    (%)
fuente                                                            
PRONIED-UGM-PREVENTIVO                              397,965  92.9%
PRONIED-UGM-PREVENTIVO / Análisis BI GR/GL del ...    8,042   1.9%
PRONIED-UGM-PREVENTIVO / Análisis IOARR GR/GL d...    6,759   1.6%
PRONIED-UGM-PREVENTIVO / Análisis IOARR GN del ...    3,818   0.9%
Análisis BI GR/GL del EV-PNIE                         2,796   0.7%
...                                                     ...    ...
PRONIED-UGRD-MBR / PRONIED-UGM-PREVENTIVO / Aná...        1   0.0%
PRONIED-UGRD-ME / PRONIED-UGM-PREVENTIVO / Anál...        1   0.0%
PRONIED-UGME-SM / PRONIED-UGM-PREVENTIVO / FONC...        1   0.0%
PRONIED-UGME-SM / Análisis IOARR GN del EV-PNIE           1   0.0%
TOTAL                                               428,337   100%

[100 rows x 2 columns] 



In [ ]:
df1 = df_result.copy()

df1 = df1[['codlocal', 'anio_culm',
           'GI1_1', 'GI1_2', 'GI1_3', 'GI1_4', 'GI1_5', 
           'GI2_1', 'GI2_2',
           'GI3_1', 'GI3_2', 'GI3_3', 'GI3_4', 
           'GI4_1', 'GI4_2', 'GI4_3', 'GI4_4',
           'GI5_1', 'GI5_2', 
           'CUI', 'fuente']]

# Exportando TODAS las intervenciones identificadas 
freq(df1, 'anio_culm')
df1.to_csv('data/procesadas/intervenciones_TOTAL.csv', index=False)

# Considerando solo las intervenciones desde 2017
df1 = df1[df1['anio_culm'] >= 2017]
freq(df1, 'anio_culm')
df1.to_csv('data/procesadas/intervenciones.csv', index=False)

              Freq    (%)
anio_culm                
2023        53,748  12.5%
2024        53,715  12.5%
2022        53,004  12.4%
2021        52,962  12.4%
2025        52,477  12.3%
2020        49,972  11.7%
2019        49,077  11.5%
2018        35,257   8.2%
2017        26,035   6.1%
2009           414   0.1%
2013           314   0.1%
2014           278   0.1%
2012           229   0.1%
2010           229   0.1%
2016           154   0.0%
2011           154   0.0%
2015           131   0.0%
2008           104   0.0%
2007            38   0.0%
2006            34   0.0%
2005             9   0.0%
2004             2   0.0%
TOTAL      428,337   100% 

              Freq    (%)
anio_culm                
2023        53,748  12.6%
2024        53,715  12.6%
2022        53,004  12.4%
2021        52,962  12.4%
2025        52,477  12.3%
2020        49,972  11.7%
2019        49,077  11.5%
2018        35,257   8.3%
2017        26,035   6.1%
TOTAL      426,247   100% 



In [ ]:
df[df['codlocal'].duplicated(keep=False)].sort_values(by='codlocal')